## SPL Section Information Extraction

### 0. Phase 00 - SPL Extraction [WIP]

In [4]:
import pandas as pd
import re
# all_spl = pd.read_csv('data/fdalabel_all/fdalabel-query-380928.csv')
# # filter out "Unapproved homeopathic" from Marketing category
# all_spl_no_homeo = all_spl[all_spl["Marketing Category"]!="Unapproved homeopathic"]
# spl_for_testing = all_spl_no_homeo.sample(n=382)
# spl_for_testing = spl_for_testing["SET ID"].reset_index(drop=True)
# spl_for_testing = spl_for_testing.to_frame()

In [3]:
# select the first 10 rows of results/contra_gold.csv and save to results/contra_gold_sample.csv
contra_gold = pd.read_csv('results/contra_gold.csv')
contra_gold_sample = contra_gold.head(100)
contra_gold_sample.to_csv('results/contra_gold_100.csv', index=False)

In [ ]:
N = 3
nth_index = N - 1 # Adjust for 0-based indexing

# 1. Get all unique values in the 'Name' column
unique_values = df['Name'].unique() #

# 2. Select the Nth unique value
if nth_index < len(unique_values):
    target_value = unique_values[nth_index]
    print(f"The {N}th unique value is: {target_value}")

    # 3. Filter the DataFrame to get the row(s) with this value
    # To get only the first row where this value appears
    target_row = df[df['Name'] == target_value].iloc[0]
    
    print("\nThe corresponding row is:")
    print(target_row)
else:
    print(f"Error: N ({N}) is greater than the number of unique values ({len(unique_values)})")

In [15]:
gold_df_raw = pd.read_csv('data/Manual_annotation_final_WM.csv')
keep_cols = ['SET ID','contra_text','Annotated Contraindications', 'Contraindications (Anaphylaxis Manual)']
gold_df_raw = gold_df_raw[keep_cols]
gold_df_raw = gold_df_raw.dropna(subset=['SET ID', 'contra_text'])
gold_df_raw = gold_df_raw.rename(columns={'SET ID':'SPL_SET_ID', 'contra_text': 'CONTRA_TEXT','Annotated Contraindications':'Annotated_1', 'Contraindications (Anaphylaxis Manual)':'Annotated_2' })

/tmp/ipykernel_763568/1861131974.py:1: DtypeWarning: Columns (1,2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  gold_df_raw = pd.read_csv('data/Manual_annotation_final_WM.csv')


In [16]:
gold_df_raw

,SPL_SET_ID,CONTRA_TEXT,Annotated_1,Annotated_2
0,285e60f8-2404-47d0-a171-a4f075d6f418,OPANA is contraindicated in patients with a kn...,Respiratory depression;\nAcute or severe bronc...,Hypersensitivity to oxymorphone;\nHypersensiti...
3,b6b45b87-aa53-4d3a-be59-f3d3b0774a7b,Chlorhexidine gluconate oral rinse should not ...,NaN,Hypersensitivity to chlorhexidine gluconate;\n...
4,aa5028d6-fd0a-4cf1-b708-c93bb2a7c76a,Nonselective monoamine oxidase (MAO) inhibitor...,concomitant use with nonselective monoamine ox...,Hypersensitivity to any component of carbidopa...
5,a7b85288-9099-4c20-beb7-ee2be484c9b9,Captopril tablets are contraindicated in patie...,NaN,Hypersensitivity to the drug;\nHypersensitivit...
6,e464e9bb-48e8-4b9f-9fff-e220cfbac0c5,Allergic reactions to organic nitrates are ext...,NaN,Allergy to organic nitrates
...,...,...,...,...
377,06ade2e1-5981-4e3d-b7e9-7ff083eb4551,Metformin Hydrochloride Tablets USP are contra...,renal disease;\nrenal dysfunction;\nAcute meta...,hypersensitivity to metformin hydrochloride
378,5716b0d8-0efc-44aa-a17e-a51f24a0c8f1,Phenadoz is contraindicated for use in pediatr...,Patients less than two years of age;\nIndividu...,Individuals known to be hypersensitive or to h...
379,e602c70f-9ea3-480f-a7df-15f23c6b3307,Metoprolol succinate extended-release tablets ...,severe bradycardia;\nSecond-degree heart block...,Hypersensitivity to any component of this product
380,1e81fa46-1164-4a34-b212-dcc16f3dc2c6,"Efavirenz, Emtricitabine and Tenofovir di...",coadministration with voriconazole;\ncoadminis...,hypersensitivity to efavirenz;\nhypersensitivi...


In [4]:
splitter = re.compile(r"\s*(?:;|\r?\n)+\s*")

def split_cell(x) -> list[str]:
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    return [t.strip() for t in splitter.split(s) if t and t.strip()]

def explode_all_annotations(df: pd.DataFrame,
                            id_col="SPL_SET_ID",
                            text_col="CONTRA_TEXT",
                            ann_cols=("Annotated_1", "Annotated_2"),
                            dedupe=True) -> pd.DataFrame:
    d = df[[id_col, text_col, *ann_cols]].copy()

    # split each annotation column into lists
    for c in ann_cols:
        d[c] = d[c].apply(split_cell)

    # combine lists across Annotated_1 + Annotated_2
    d["annotation_list"] = d.apply(lambda r: sum((r[c] for c in ann_cols), []), axis=1)

    # optional dedupe while preserving order
    if dedupe:
        def dedupe_preserve(seq):
            seen = set()
            out = []
            for x in seq:
                if x not in seen:
                    seen.add(x)
                    out.append(x)
            return out
        d["annotation_list"] = d["annotation_list"].apply(dedupe_preserve)

    # explode to long: one row per annotation item
    out = d[[id_col, text_col, "annotation_list"]].explode("annotation_list", ignore_index=True)
    out = out.rename(columns={"annotation_list": "annotation"})

    # drop empties (safety)
    out["annotation"] = out["annotation"].fillna("").astype(str).str.strip()
    out = out[out["annotation"].ne("")].copy()

    # create contra_id within each SPL_SET_ID
    out["contra_idx"] = out.groupby(id_col).cumcount() + 1
    out["contra_id"] = out[id_col].astype(str) + "_" + out["contra_idx"].astype(str).str.zfill(3)

    return out[[id_col, "contra_id", text_col, "annotation"]]

# usage:
# df = pd.read_excel("your_file.xlsx")
# long_df = explode_all_annotations(df, dedupe=True)
# long_df.to_csv("contra_long.csv", index=False)


In [5]:
gold_df = explode_all_annotations(gold_df_raw, dedupe=False)
gold_df

,SPL_SET_ID,contra_id,CONTRA_TEXT,annotation
0,285e60f8-2404-47d0-a171-a4f075d6f418,285e60f8-2404-47d0-a171-a4f075d6f418_001,OPANA is contraindicated in patients with a kn...,Respiratory depression
1,285e60f8-2404-47d0-a171-a4f075d6f418,285e60f8-2404-47d0-a171-a4f075d6f418_002,OPANA is contraindicated in patients with a kn...,Acute or severe bronchial asthma
2,285e60f8-2404-47d0-a171-a4f075d6f418,285e60f8-2404-47d0-a171-a4f075d6f418_003,OPANA is contraindicated in patients with a kn...,Hypercapnia
3,285e60f8-2404-47d0-a171-a4f075d6f418,285e60f8-2404-47d0-a171-a4f075d6f418_004,OPANA is contraindicated in patients with a kn...,Paralytic ileus
4,285e60f8-2404-47d0-a171-a4f075d6f418,285e60f8-2404-47d0-a171-a4f075d6f418_005,OPANA is contraindicated in patients with a kn...,Moderate or severe hepatic impairment
...,...,...,...,...
1490,1e81fa46-1164-4a34-b212-dcc16f3dc2c6,1e81fa46-1164-4a34-b212-dcc16f3dc2c6_001,"Efavirenz, Emtricitabine and Tenofovir di...",coadministration with voriconazole
1491,1e81fa46-1164-4a34-b212-dcc16f3dc2c6,1e81fa46-1164-4a34-b212-dcc16f3dc2c6_002,"Efavirenz, Emtricitabine and Tenofovir di...",coadministration with elbasvir / grazoprevir
1492,1e81fa46-1164-4a34-b212-dcc16f3dc2c6,1e81fa46-1164-4a34-b212-dcc16f3dc2c6_003,"Efavirenz, Emtricitabine and Tenofovir di...",hypersensitivity to efavirenz
1493,1e81fa46-1164-4a34-b212-dcc16f3dc2c6,1e81fa46-1164-4a34-b212-dcc16f3dc2c6_004,"Efavirenz, Emtricitabine and Tenofovir di...",hypersensitivity to components of efavirenz


In [ ]:
gold_df.to_csv('results/contra_gold.csv')

In [6]:
import pandas as pd
gold_df = pd.read_csv('results/contra_gold.csv')
gold_df = gold_df.drop_duplicates(subset=["SPL_SET_ID"], keep="first")
gold_df

,Unnamed: 0,SPL_SET_ID,contra_id,CONTRA_TEXT,annotation
0,0,285e60f8-2404-47d0-a171-a4f075d6f418,285e60f8-2404-47d0-a171-a4f075d6f418_001,OPANA is contraindicated in patients with a kn...,Respiratory depression
8,8,b6b45b87-aa53-4d3a-be59-f3d3b0774a7b,b6b45b87-aa53-4d3a-be59-f3d3b0774a7b_001,Chlorhexidine gluconate oral rinse should not ...,Hypersensitivity to chlorhexidine gluconate
10,10,aa5028d6-fd0a-4cf1-b708-c93bb2a7c76a,aa5028d6-fd0a-4cf1-b708-c93bb2a7c76a_001,Nonselective monoamine oxidase (MAO) inhibitor...,concomitant use with nonselective monoamine ox...
14,14,a7b85288-9099-4c20-beb7-ee2be484c9b9,a7b85288-9099-4c20-beb7-ee2be484c9b9_001,Captopril tablets are contraindicated in patie...,Hypersensitivity to the drug
16,16,e464e9bb-48e8-4b9f-9fff-e220cfbac0c5,e464e9bb-48e8-4b9f-9fff-e220cfbac0c5_001,Allergic reactions to organic nitrates are ext...,Allergy to organic nitrates
...,...,...,...,...,...
1462,1473,06ade2e1-5981-4e3d-b7e9-7ff083eb4551,06ade2e1-5981-4e3d-b7e9-7ff083eb4551_001,Metformin Hydrochloride Tablets USP are contra...,renal disease
1468,1479,5716b0d8-0efc-44aa-a17e-a51f24a0c8f1,5716b0d8-0efc-44aa-a17e-a51f24a0c8f1_001,Phenadoz is contraindicated for use in pediatr...,Patients less than two years of age
1472,1483,e602c70f-9ea3-480f-a7df-15f23c6b3307,e602c70f-9ea3-480f-a7df-15f23c6b3307_001,Metoprolol succinate extended-release tablets ...,severe bradycardia
1479,1490,1e81fa46-1164-4a34-b212-dcc16f3dc2c6,1e81fa46-1164-4a34-b212-dcc16f3dc2c6_001,"Efavirenz, Emtricitabine and Tenofovir di...",coadministration with voriconazole


### Extraction of sample for Manual Annotation

In [1]:
from VaxMapper.src.utils.dailymed import extract_section
CONTRA_Loinc = "34070-3"
ACTIVE_loinc = "55106-9"

def _safe_get_contra(sid):
    try:
        res = extract_section(sid, [CONTRA_Loinc])
        if res.get("found"):
            return res["sections"].get(CONTRA_Loinc, {}).get("section_text", "")
        return ""
    except Exception:
        return ""

In [6]:
tt = extract_section(setid="9dfe774a-2c7a-4e10-bedd-cde64ad28676", loinc_code=[CONTRA_Loinc])

In [8]:
ty = extract_section(setid="40fa228f-9bfc-fc71-e063-6394a90a9ab6", loinc_code=[CONTRA_Loinc])


In [9]:
ty

{'setid': '40fa228f-9bfc-fc71-e063-6394a90a9ab6',
 'product_name': 'EvexiTHROID',
 'found': True,
 'sections': {'34070-3': {'section_xml': '<section xmlns="urn:hl7-org:v3" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" ID="id_link_40fb0a47-ee71-fb99-e063-6394a90acef6">\n  <id root="40fb0a47-ee70-fb99-e063-6394a90acef6"/>\n  <code code="34070-3" codeSystem="2.16.840.1.113883.6.1" displayName="CONTRAINDICATIONS SECTION"/>\n  <title>CONTRAINDICATIONS</title>\n  <text>\n    <paragraph>Thyroid hormone preparations are generally contraindicated in patients with diagnosed but as yet uncorrected adrenal cortical insufficiency, untreated thyrotoxicosis, and apparent hypersensitivity to any of their active or extraneous constituents. There is no well-documented evidence from the literature, however, of true allergic or idiosyncratic reactions to thyroid hormone.</paragraph>\n  </text>\n  <effectiveTime value="20251012"/>\n</section>\n',
   'section_text': 'Thyroid hormone preparations are

In [7]:
tt

{'setid': '9dfe774a-2c7a-4e10-bedd-cde64ad28676',
 'product_name': 'Eszopiclone',
 'found': True,
 'sections': {'34070-3': {'section_xml': '<section xmlns="urn:hl7-org:v3" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" ID="Section_4">\n  <id root="3aaedc44-725b-b21e-e063-6394a90a2a93"/>\n  <code code="34070-3" codeSystem="2.16.840.1.113883.6.1" displayName="CONTRAINDICATIONS SECTION"/>\n  <title>4 CONTRAINDICATIONS</title>\n  <text>\n    <paragraph>\n      <content styleCode="xmChange">•\xa0Eszopiclonetablets are contraindicated in patients who have experienced complex sleep behaviors after taking eszopiclone tablets [\n \n   <content styleCode="italics">see Warnings and Precautions (\n  \n    <linkHtml href="#Section_5.8">5.1</linkHtml>)\n \n   </content>]. \n   <br/>  •\xa0Eszopiclone tablets are contraindicated in patients with known hypersensitivity to eszopiclone. Hypersensitivity reactions include anaphylaxis and angioedema [\n \n   <content styleCode="italics">see Warning

In [36]:
spl_for_testing['contra_text'] = spl_for_testing["SET ID"].apply(_safe_get_contra)

In [37]:
spl_for_testing

,SET ID,contra_text
0,285e60f8-2404-47d0-a171-a4f075d6f418,OPANA is contraindicated in patients with a kn...
1,89544705-5d9d-4595-8870-fa46c9c14af2,
2,88f80e1e-d6e3-4789-abe1-f62db9ca366b,
3,b6b45b87-aa53-4d3a-be59-f3d3b0774a7b,Chlorhexidine gluconate oral rinse should not ...
4,aa5028d6-fd0a-4cf1-b708-c93bb2a7c76a,Nonselective monoamine oxidase (MAO) inhibitor...
...,...,...
377,06ade2e1-5981-4e3d-b7e9-7ff083eb4551,Metformin Hydrochloride Tablets USP are contra...
378,5716b0d8-0efc-44aa-a17e-a51f24a0c8f1,Phenadoz is contraindicated for use in pediatr...
379,e602c70f-9ea3-480f-a7df-15f23c6b3307,Metoprolol succinate extended-release tablets ...
380,1e81fa46-1164-4a34-b212-dcc16f3dc2c6,"Efavirenz, Emtricitabine and Tenofovir di..."


In [38]:
spl_for_testing.to_csv('results/spl382.csv')

### 1. Phase 01 - Identification

#### 1.1 Contraindication sections [SPL]

In [ ]:
# # 12-08-2025 (Contraindications DF as is. Some contra texts are blank. Needs to be fixed)
# import pandas as pd
# # rx_spl_df = pd.read_csv('results/vaccine_spl_contra_adverse_sections.csv')
# rx_spl_df = pd.read_csv('results/spl382.csv')

#### 1.2 Contraindication Identification [LLM]

In [66]:
from VaxMapper.src.llm import load_model_local, build_messages_from_iter, extract_json
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import torch
import pickle
torch.set_float32_matmul_precision('high')

In [22]:
import datetime
current_date = datetime.datetime.now().date()

In [67]:
model = load_model_local("google/medgemma-27b-text-it")

Loading checkpoint shards:   0%|          | 0/11 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [32]:
# del model
torch.cuda.empty_cache()

In [13]:
## MINIMAL CONTRAINDICATIONS EXTRACTION PROMPT
system = """
You are a biomedical NLP assistant that identifies CONTRAINDICATIONS in regulatory drug or vaccine documents.
Your ONLY task is to find text spans that express a contraindication and return them as JSON with character offsets.

--------------------
DEFINITIONS
--------------------
A contraindication is a condition or situation where the product SHOULD NOT be used or administered.

An ATOMIC contraindication is the smallest indivisible contraindicated condition or situation.
Each atomic contraindication must be expressed as its own item.

--------------------
CRITICAL INSTRUCTION: SPLITTING COORDINATED CONTRAINDICATIONS
--------------------
If a contraindication statement contains coordination (e.g., "or", "and", commas, lists), you MUST ALWAYS split it into MULTIPLE atomic contraindications whenever each item could stand alone.

Examples:

Text:
"known hypersensitivity to drug A or to any of the other ingredients in drug A, or with known hypersensitivity to drug B analogs, including or such as drug C"

You MUST extract THREE separate contraindications:
1) hypersensitivity to drug A
2) hypersensitivity to ingredients of drug A
3) hypersensitivity to drug B

DO NOT add a separate contraindication for the example drug (drug C)

Text:
"moderate or severe condition X is a contraindication"

You MUST extract TWO separate contraindications:
1) moderate condition X
2) severe condition X


Do NOT merge coordinated items into a single generalized condition.
Do NOT abstract away specific substances or categories.

--------------------
OUTPUT FORMAT
--------------------
You MUST return ONLY a single line of MINIFIED JSON. 
Do NOT include any markdown code fences (no ```).
Do NOT include any explanations, reasoning, or extra text.
Do NOT include the word "json".
Do NOT include trailing commentary.

The JSON must end with the exact token:

<<END_JSON>>

Example format (illustrative only):
{{"items": [{{"span_text": "<verbatim contraindication span>","condition_text": "<short phrase naming the condition or situation>"}}]}}<<END_JSON>>

Rules:
- "span_text" must be copied verbatim from the input text for that span.
- "condition_text" should be a concise, normalized description of the contraindicated condition.
- If there are no contraindications in the text, return: {{"items": []}}
- The JSON must be valid.
- The JSON must be minified (no newlines, no indentation).
- The JSON must immediately be followed by <<END_JSON>>.
- Nothing may appear after <<END_JSON>>.

--------------------
TASK
--------------------
The user will provide a block of TEXT (e.g., the CONTRAINDICATIONS section of an SPL).

Carefully read the TEXT and output ONLY the JSON object described above.
Do not include any explanations, comments, or additional text outside the JSON. Do not show your reasoning. Provide only the final answer.
"""

user = """
Here is the CONTRAINDICATIONS section from a vaccine SPL document: \n{text}
"""

In [17]:
# sections = rx_spl_df['contra_text'].tolist()[:50]
# msgs = build_messages_from_iter(system, user, sections)
sections = gold_df_raw['CONTRA_TEXT'].tolist()[:2]
msgs = build_messages_from_iter(system, user, sections)

In [18]:
out = [model.generate(m, max_new_tokens=512) for m in msgs]

In [19]:
json_list = []
for i in out:
    res = extract_json(i)
    json_list.append(res)

In [49]:
out_name = f'results/{current_date}_all_spl_contra_extracted.pkl'
with open(out_name, 'wb') as f:
    pickle.dump(json_list, f)

In [ ]:
# read picked file
import pickle
out_name = "results/{current_date}_vaccine_spl_contra_extracted.pkl"
with open(out_name, 'rb') as f:
    json_list = pickle.load(f)

In [22]:
out[0]

'{"items":[{"span_text":"known hypersensitivity to oxymorphone","condition_text":"hypersensitivity to oxymorphone"},{"span_text":"to any of the other ingredients in OPANA","condition_text":"hypersensitivity to ingredients of OPANA"},{"span_text":"with known hypersensitivity to morphine analogs such as codeine","condition_text":"hypersensitivity to morphine analogs"},{"span_text":"respiratory depression","condition_text":"respiratory depression"},{"span_text":"acute or severe bronchial asthma","condition_text":"acute bronchial asthma"},{"span_text":"severe bronchial asthma","condition_text":"severe bronchial asthma"},{"span_text":"hypercarbia","condition_text":"hypercarbia"},{"span_text":"paralytic ileus","condition_text":"paralytic ileus"},{"span_text":"moderate or severe hepatic impairment","condition_text":"moderate hepatic impairment"},{"span_text":"severe hepatic impairment","condition_text":"severe hepatic impairment"}]}<<END_JSON>>thought\nThe user wants me to extract contraindic

In [20]:
json_list[0]

{'items': [{'span_text': 'known hypersensitivity to oxymorphone',
   'condition_text': 'hypersensitivity to oxymorphone'},
  {'span_text': 'to any of the other ingredients in OPANA',
   'condition_text': 'hypersensitivity to ingredients of OPANA'},
  {'span_text': 'with known hypersensitivity to morphine analogs such as codeine',
   'condition_text': 'hypersensitivity to morphine analogs'},
  {'span_text': 'respiratory depression',
   'condition_text': 'respiratory depression'},
  {'span_text': 'acute or severe bronchial asthma',
   'condition_text': 'acute bronchial asthma'},
  {'span_text': 'severe bronchial asthma',
   'condition_text': 'severe bronchial asthma'},
  {'span_text': 'hypercarbia', 'condition_text': 'hypercarbia'},
  {'span_text': 'paralytic ileus', 'condition_text': 'paralytic ileus'},
  {'span_text': 'moderate or severe hepatic impairment',
   'condition_text': 'moderate hepatic impairment'},
  {'span_text': 'severe hepatic impairment',
   'condition_text': 'severe he

In [21]:
for x in json_list[0].get('items'):
    print(x['condition_text'])

hypersensitivity to oxymorphone
hypersensitivity to ingredients of OPANA
hypersensitivity to morphine analogs
respiratory depression
acute bronchial asthma
severe bronchial asthma
hypercarbia
paralytic ileus
moderate hepatic impairment
severe hepatic impairment


#### 1.3 Eval of Contraindication Identification [LLM]

##### Temp: Testing of the evluation protocol

In [31]:
import os
import re
import json
import glob
import argparse
from typing import List, Dict, Any, Tuple
import pandas as pd
from VaxMapper.src.llm import extract_json


def read_predictions_jsonl_files(pred_files: List[str], id_field_candidates=("spl_set_id", "SPL_SET_ID","item_index")) -> pd.DataFrame:
    """
    Returns DataFrame with columns:
      - SPL_SET_ID
      - pred_terms : list[str]
    """
    rows = []
    for path in pred_files:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rec = json.loads(line)

                spl_id = None
                for k in id_field_candidates:
                    if k in rec and rec[k] is not None:
                        spl_id = str(rec[k])
                        break
                if spl_id is None:
                    raise ValueError(f"Prediction record missing SPL_SET_ID/spl_set_id in file {path}")

                raw = rec.get("raw_output") or {}
                parsed = extract_json(raw)
                items = parsed.get("items") or []
                pred_terms = []
                for it in items:
                    if isinstance(it, dict):
                        ct = it.get("condition_text")
                        if ct:
                            pred_terms.append(str(ct))

                rows.append({"SPL_SET_ID": spl_id, "pred_terms": pred_terms})
    return pd.DataFrame(rows)

pred_files = ["results/contra_ie1/out_gpu0.jsonl","results/contra_ie1/out_gpu1.jsonl"]
pred_df = read_predictions_jsonl_files(pred_files)
pred_df

,SPL_SET_ID,pred_terms
0,285e60f8-2404-47d0-a171-a4f075d6f418,"[hypersensitivity to oxymorphone, hypersensiti..."
1,aa5028d6-fd0a-4cf1-b708-c93bb2a7c76a,"[use with nonselective MAO inhibitors, hyperse..."
2,e464e9bb-48e8-4b9f-9fff-e220cfbac0c5,[allergy to nitroglycerin]
3,49245aec-9a0f-459e-e054-00144ff8d46c,"[hypersensitivity to tamsulosin hydrochloride,..."
4,829e8675-959a-4fba-8e2b-464aaec930fa,"[hypersensitivity to the drug, hypersensitivit..."
...,...,...
339,d7b7c365-0093-4c98-84b1-e1742ce3a08f,[Hypersensitivity to baclofen]
340,a262d7c6-2aa9-433c-b801-d8b2dcd68a0f,"[hypersensitivity to pregabalin, hypersensitiv..."
341,06ade2e1-5981-4e3d-b7e9-7ff083eb4551,"[Renal disease or renal dysfunction, abnormal ..."
342,e602c70f-9ea3-480f-a7df-15f23c6b3307,"[severe bradycardia, second- or third- degree ..."


In [28]:
test_spl = '00c486b0-bd7b-4898-834d-8c656e5e73cb'
# test_spl = '092f1cea-edf2-4536-b378-34b28e281b83'
pred_terms = pred_df[pred_df['SPL_SET_ID']==test_spl]['pred_terms'].explode().to_list()
pred_terms


['Hypersensitivity to GnRH',
 'Hypersensitivity to GnRH agonist analogs',
 'Hypersensitivity to leuprolide acetate',
 'Hypersensitivity to excipients in LUPRON DEPOT 3.75 mg',
 'Undiagnosed abnormal uterine bleeding',
 'Pregnancy']

In [17]:
gold_df = pd.read_csv('results/contra_gold.csv')
gold_terms = gold_df[gold_df['SPL_SET_ID']==test_spl]['annotation'].to_list()
gold_terms

['For example',
 'Increased plasma concentrations of some of these drugs can lead to QT prolongation and ventricular tachyarrhythmias including occurrences of torsades de pointes',
 'A potentially fatal arrhythmia',
 'Coadministration of a number of CYP3A4 substrates',
 'Patients with acute',
 'Patients chronic liver disease',
 'Hypersensitivity to the drug']

### 2. Phase 02 - Similarity

#### 2.1 Lexical Similarity

In [21]:
from VaxMapper.src.utils.elastisearch_utils import run_elasticsearch, stop_elasticsearch, get_es_client, create_index, doc_actions, bulk_index
import pandas as pd

In [10]:
run_elasticsearch()
es = get_es_client()

Starting Elasticsearch from /data/wmanuel3/VaxMapperRepo/esdata/elasticsearch-9.0.0/bin/elasticsearch: on port 9200...
Elasticsearch is running: 9.0.0


In [43]:
# read txt file as dataframe
snomed_con_df = pd.read_csv('snomed_us_source/sct2_Concept_Snapshot_US1000124_20250901.txt', sep='\t')
snomed_des_df = pd.read_csv('snomed_us_source/sct2_Description_Snapshot-en_US1000124_20250901.txt', sep='\t')

In [44]:
snomed_active_con = snomed_con_df[snomed_con_df['active']==1]
snomed_des_df = snomed_des_df[(snomed_des_df["conceptId"].isin(snomed_active_con["id"])) & (snomed_des_df["active"]==1)]
snomed_des_df = snomed_des_df[['conceptId', 'term', 'typeId']]
snomed_des_df

,conceptId,term,typeId
0,126813005,Neoplasm of anterior aspect of epiglottis,900000000000013009
1,126814004,Neoplasm of junctional region of epiglottis,900000000000013009
2,126815003,Neoplasm of lateral wall of oropharynx,900000000000013009
3,126816002,Neoplasm of posterior wall of oropharynx,900000000000013009
4,126817006,Neoplasm of esophagus,900000000000013009
...,...,...,...
1696622,900000000000550004,Definition,900000000000013009
1696623,900000000000550004,Definition (core metadata concept),900000000000003001
1696624,625544171000132109,Disorder of pancreatic stent,900000000000013009
1696625,933316181000036101,Vorasidenib (substance),900000000000003001


In [45]:
concept_df = snomed_des_df[snomed_des_df['typeId'] == 900000000000003001][['conceptId', 'term']] # FSN
synonym_df = snomed_des_df[snomed_des_df['typeId'] == 900000000000013009][['conceptId', 'term']] # SYN
syn_agg_df = synonym_df.groupby('conceptId')['term'].apply(list).reset_index()

In [1]:
!nvidia-smi

Tue Mar  3 13:45:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.51.03              Driver Version: 575.51.03      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:27:00.0 Off |                    0 |
| N/A   24C    P0             59W /  400W |      92MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [46]:
import re 
def extract_semantic_tag(term: str) -> str:
    match = re.search(r"\(([^)]+)\)\s*$", term)
    return match.group(1).lower() if match else ""

In [47]:
concept_df["semantic_tag"] = concept_df["term"].apply(extract_semantic_tag)
concept_df

,conceptId,term,semantic_tag
534,127362006,Previous pregnancies (finding),finding
181775,125587004,Superficial injury (disorder),disorder
181830,125643001,Open wound (disorder),disorder
181852,125665001,Crushing injury (disorder),disorder
181853,125666000,Burn (disorder),disorder
...,...,...,...
1696617,900000000000547002,Relationship inactivation indicator attribute ...,foundation metadata concept
1696618,900000000000548007,Preferred (foundation metadata concept),foundation metadata concept
1696620,900000000000549004,Acceptable (foundation metadata concept),foundation metadata concept
1696623,900000000000550004,Definition (core metadata concept),core metadata concept


In [48]:
synonym_df

,conceptId,term
0,126813005,Neoplasm of anterior aspect of epiglottis
1,126814004,Neoplasm of junctional region of epiglottis
2,126815003,Neoplasm of lateral wall of oropharynx
3,126816002,Neoplasm of posterior wall of oropharynx
4,126817006,Neoplasm of esophagus
...,...,...
1696619,900000000000548007,Preferred
1696621,900000000000549004,Acceptable
1696622,900000000000550004,Definition
1696624,625544171000132109,Disorder of pancreatic stent


In [49]:
def make_terms_df(concept_df: pd.DataFrame, synonym_df: pd.DataFrame) -> pd.DataFrame:
    # concept_df: concept_id unique, preferredTerm
    pref = concept_df[["conceptId", "term"]].rename(columns={"term": "term_text"})
    pref["term_type"] = "preferred"

    # synonym_df: concept_id non-unique, synonymTerm
    syn = synonym_df[["conceptId", "term"]].rename(columns={"term": "term_text"})
    syn["term_type"] = "synonym"

    terms_df = pd.concat([pref, syn], ignore_index=True)

    # drop empties + dedupe within concept
    terms_df["term_text"] = terms_df["term_text"].astype(str)
    terms_df = terms_df[terms_df["term_text"].str.strip() != ""]
    terms_df = terms_df.drop_duplicates(subset=["conceptId", "term_text"])

    return terms_df


In [50]:
terms_df = make_terms_df(concept_df, synonym_df)
terms_df

,conceptId,term_text,term_type
0,127362006,Previous pregnancies (finding),preferred
1,125587004,Superficial injury (disorder),preferred
2,125643001,Open wound (disorder),preferred
3,125665001,Crushing injury (disorder),preferred
4,125666000,Burn (disorder),preferred
...,...,...,...
1017043,900000000000548007,Preferred,synonym
1017044,900000000000549004,Acceptable,synonym
1017045,900000000000550004,Definition,synonym
1017046,625544171000132109,Disorder of pancreatic stent,synonym


In [26]:
from VaxMapper.src.utils.snomed_utils import load_snomed_dataframes
out = load_snomed_dataframes(snomed_source_dir='snomed_source')
out["terms_df"]

FileNotFoundError: No RF2 file found in 'snomed_source' matching 'sct2_Concept_Snapshot_*.txt'.

In [57]:
# get non unique conceptId in synonym_df
synonym_df['conceptId'].value_counts()

conceptId
79962008              34
128822004             29
86986002              27
4887000               24
398796005             23
                      ..
900000000000518009     1
900000000000520007     1
900000000000523009     1
900000000000524003     1
900000000000525002     1
Name: count, Length: 382168, dtype: int64

In [21]:
snomed_complete_df = pd.merge(concept_df, syn_agg_df, on='conceptId', how='outer')

In [22]:
snomed_complete_df.rename(columns={'term_x': 'preferredTerm', 'term_y': 'synonymTerms'}, inplace=True)
# snomed_complete_df

In [23]:
snomed_complete_df

,conceptId,preferredTerm,semantic_tag,synonymTerms
0,101009,Quilonia ethiopica (organism),organism,[Quilonia ethiopica]
1,102002,Hemoglobin Okaloosa (substance),substance,"[Hemoglobin Okaloosa, Hb 48(CD7), Leu-arg, Hae..."
2,103007,Squirrel fibroma virus (organism),organism,[Squirrel fibroma virus]
3,104001,Excision of lesion of patella (procedure),procedure,"[Excision of lesion of patella, Local excision..."
4,107008,Structure of fetal part of placenta (body stru...,body structure,"[Fetal part of placenta, Structure of fetal pa..."
...,...,...,...,...
382165,997199941000119102,Fluoroscopic venography of left limb with cont...,procedure,[Fluoroscopic venography of left extremity wit...
382166,998010041000087101,Legionella hackeliae serogroup 2 (organism),organism,[Legionella hackeliae serogroup 2]
382167,998010741000119109,Sensation of foreign body in bilateral eyes (f...,finding,"[Sensation of foreign body in both eyes, Sensa..."
382168,999480551000087103,Aspergillus japonicus (organism),organism,[Aspergillus japonicus]


In [ ]:
260413007

In [24]:
snomed_ct_settings = {
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 1,
      "similarity": {
        "default": {
          "type": "BM25",
          "k1": 1.2, #previously 0.9
          "b": 0.75 # previously 0.4
        }
      }
    },
    "analysis": {
      "analyzer": {
        "snomed_text": {
          "type": "custom",
          "tokenizer": "standard",
          "filter": ["lowercase","asciifolding"]
        }
      }
    }
  },
  "mappings": {
    "properties": {
      "conceptId": {
        "type": "keyword"
      },
      "preferredTerm": {
        "type": "text",
        "analyzer": "snomed_text",
        "search_analyzer": "snomed_text",
        "copy_to": ["all_terms"]
      },
      "synonyms": {
        "type": "text",
        "analyzer": "snomed_text",
        "search_analyzer": "snomed_text",
        "copy_to": ["all_terms"]
      },
      "all_terms": {
        "type": "text",
        "analyzer": "snomed_text",
        "search_analyzer": "snomed_text"
      },
      "semantic_tag":{
          "type": "keyword"
      }
    }
  }
}


In [25]:
create_index(es, 'snomed_ct_es_index', snomed_ct_settings,delete_if_exists=True)

Deleting existing index: snomed_ct_es_index
Creating index 'snomed_ct_es_index'...


In [26]:
field_map = {
    "conceptId": "conceptId", # df col -> ES field
    "preferredTerm": "preferredTerm",
    "synonymTerms": "synonyms",
    "semantic_tag": "semantic_tag"
}

In [27]:
bulk_index(es = es, df = snomed_complete_df, id_col='conceptId', index_name = 'snomed_ct_es_index', field_map = field_map)

Failed to index: {'index': {'_index': 'snomed_ct_es_index', '_id': '260413007', 'status': 400, 'error': {'type': 'document_parsing_exception', 'reason': '[1:77] failed to parse: [1:80] Non-standard token \'NaN\': enable `JsonReadFeature.ALLOW_NON_NUMERIC_NUMBERS` to allow\n at [Source: (byte[])"{"conceptId":260413007,"preferredTerm":"None (qualifier value)","synonyms":[NaN],"semantic_tag":"qualifier value"}"; line: 1, column: 80]', 'caused_by': {'type': 'x_content_parse_exception', 'reason': '[1:80] Non-standard token \'NaN\': enable `JsonReadFeature.ALLOW_NON_NUMERIC_NUMBERS` to allow\n at [Source: (byte[])"{"conceptId":260413007,"preferredTerm":"None (qualifier value)","synonyms":[NaN],"semantic_tag":"qualifier value"}"; line: 1, column: 80]', 'caused_by': {'type': 'json_parse_exception', 'reason': 'Non-standard token \'NaN\': enable `JsonReadFeature.ALLOW_NON_NUMERIC_NUMBERS` to allow\n at [Source: (byte[])"{"conceptId":260413007,"preferredTerm":"None (qualifier value)","synonyms":[

#### 2.1 Semantic Similarity

In [ ]:
# use each synonym as a seperate entity 

In [28]:
from VaxMapper.src.utils.embedding_utils import load_ST_model, build_and_save_dense_index, maybe_move_index_to_gpu
import faiss

In [ ]:
# snomed_complete_df["conceptId"] = snomed_complete_df["conceptId"].astype("int64")
# meta_df = snomed_complete_df.set_index('conceptId')

In [ ]:
# meta_df

,preferredTerm,synonymTerms
conceptId,,
101009,Quilonia ethiopica (organism),[Quilonia ethiopica]
102002,Hemoglobin Okaloosa (substance),"[Hemoglobin Okaloosa, Hb 48(CD7), Leu-arg, Hae..."
103007,Squirrel fibroma virus (organism),[Squirrel fibroma virus]
104001,Excision of lesion of patella (procedure),"[Excision of lesion of patella, Local excision..."
107008,Structure of fetal part of placenta (body stru...,"[Fetal part of placenta, Structure of fetal pa..."
...,...,...
997199941000119102,Fluoroscopic venography of left limb with cont...,[Fluoroscopic venography of left extremity wit...
998010041000087101,Legionella hackeliae serogroup 2 (organism),[Legionella hackeliae serogroup 2]
998010741000119109,Sensation of foreign body in bilateral eyes (f...,"[Sensation of foreign body in both eyes, Sensa..."


In [29]:
st_model = load_ST_model('tavakolih/all-MiniLM-L6-v2-pubmed-full', device='cuda')
# del st_model

Loading model tavakolih/all-MiniLM-L6-v2-pubmed-full...
Model tavakolih/all-MiniLM-L6-v2-pubmed-full loaded to cuda:0 successfully.


In [ ]:
# index = build_and_save_dense_index(
#     df=meta_df,
#     model=st_model,
#     label_column='preferredTerm',
#     batch_size=512,
#     normalize=True,
#     use_gpu_for_queries=True,          # GPU for runtime search if available
#     save_index=True,
#     index_filename=f'results/{current_date}_snomed_faiss_index.bin'
)

Encoding 382170 texts from column 'preferredTerm'...


Batches:   0%|          | 0/747 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [11]:
faiss_index = build_and_save_dense_index(
    df=terms_df,
    model=st_model,
    text_column="term_text",
    id_column="conceptId",
    batch_size=256,
    normalize=True,
    use_gpu_for_queries=True,
    save_index=True,
    index_filename="results/snomed_terms_dense_test.bin",
)


Encoding 1017048 texts from column 'term_text'...


Batches:   0%|          | 0/3973 [00:00<?, ?it/s]

Index built with 1017048 vectors mapped to 382170 unique concept IDs.
Saving CPU index to 'results/snomed_terms_dense_test.bin'...
Index saved successfully to 'results/snomed_terms_dense_test.bin'.
Moving FAISS index to GPU (device 0)...
Index moved to GPU successfully.


#### 2.2 Hybrid Ranking (RRF)

In [30]:
from VaxMapper.src.utils.search_utils import search_query
#ToDo: Rename search_contraindication to somethine generic line search_query or something

In [31]:
# cpu_index = faiss.read_index("results/2025-12-16_snomed_faiss_index.bin")
cpu_index = faiss.read_index("results/snomed_terms_dense_test.bin")
faiss_index = maybe_move_index_to_gpu(cpu_index)
# del cpu_index, faiss_index

Moving FAISS index to GPU (device 0)...
Index moved to GPU successfully.


##### 2.2.1 Sanity Check

In [32]:
concept_meta_df = concept_df.set_index("conceptId")
concept_meta_df

,term,semantic_tag
conceptId,,
127362006,Previous pregnancies (finding),finding
125587004,Superficial injury (disorder),disorder
125643001,Open wound (disorder),disorder
125665001,Crushing injury (disorder),disorder
125666000,Burn (disorder),disorder
...,...,...
900000000000547002,Relationship inactivation indicator attribute ...,foundation metadata concept
900000000000548007,Preferred (foundation metadata concept),foundation metadata concept
900000000000549004,Acceptable (foundation metadata concept),foundation metadata concept


In [153]:
# query = json_list[0]['items'][0]['condition_text']
query = 'oxymorphone'
query

'oxymorphone'

In [154]:
hits = search_query(
    query_text=query,
    model=st_model,
    faiss_index=faiss_index,
    concept_meta_df=concept_meta_df,
    es=es,
    bm25_index="snomed_ct_es_index",
    label_column="term", #column name of meta_df used to get concept label
    meta_id_column=None,          # use meta_df.index as id (needs to be removed. Non functional atm)
    bm25_text_field="all_terms", #
    bm25_id_field="conceptId",
    bm25_label_field="preferredTerm",
    k_dense=30,
    k_bm25=30,
    k_final=20,
)

In [155]:
for h in hits:
    print(h)

{'id': 24751001, 'label': 'Oxymorphone (substance)', 'fused': 0.03278688524590164, 'from': 'bm25,dense', 'semantic_tag': 'substance'}
{'id': 69899006, 'label': 'Oxymorphone hydrochloride (substance)', 'fused': 0.031754032258064516, 'from': 'bm25,dense', 'semantic_tag': 'substance'}
{'id': 725694003, 'label': 'Oxabolone (substance)', 'fused': 0.014285714285714285, 'from': 'dense', 'semantic_tag': 'substance'}
{'id': 126101002, 'label': 'Oxymetholone (substance)', 'fused': 0.014084507042253521, 'from': 'dense', 'semantic_tag': 'substance'}
{'id': 1354685003, 'label': 'Glyoxal (substance)', 'fused': 0.013888888888888888, 'from': 'dense', 'semantic_tag': 'substance'}
{'id': 88020002, 'label': 'Ketoacid (substance)', 'fused': 0.0136986301369863, 'from': 'dense', 'semantic_tag': 'substance'}
{'id': 96111004, 'label': 'Oxibendazole (substance)', 'fused': 0.013513513513513514, 'from': 'dense', 'semantic_tag': 'substance'}
{'id': 708825002, 'label': 'Oteracil (substance)', 'fused': 0.0133333333

In [56]:
### POST COORDD 

import pandas as pd
from collections import defaultdict, deque
from typing import Dict, Set, Optional, List

ISA = "116680003"

def read_rf2_txt(path: str, usecols: Optional[List[str]] = None) -> pd.DataFrame:
    return pd.read_csv(path, sep="\t", dtype=str, usecols=usecols, low_memory=False)

def load_is_a_parent_map(relationship_snapshot_path: str, *, use_inferred: bool = True) -> Dict[str, Set[str]]:
    """
    Build sourceId -> set(parentId) from IS-A relationships.
    For hierarchy traversal, inferred IS-A is typically OK.

    relationship snapshot columns needed:
      active, sourceId, destinationId, typeId, characteristicTypeId
    """
    rel = read_rf2_txt(
        relationship_snapshot_path,
        usecols=["active", "sourceId", "destinationId", "typeId", "characteristicTypeId"]
    )
    rel = rel[(rel["active"] == "1") & (rel["typeId"] == ISA)].copy()

    if use_inferred:
        INFERRED = "900000000000011006"
        rel = rel[rel["characteristicTypeId"] == INFERRED]
    # else: use stated if you prefer

    parents = defaultdict(set)
    for r in rel.itertuples(index=False):
        parents[r.sourceId].add(r.destinationId)
    return dict(parents)

def load_mrcm_attribute_domain(mrcm_attr_path: str) -> pd.DataFrame:
    """
    Typical columns include:
      id, effectiveTime, active, moduleId, refsetId, referencedComponentId,
      domainId, grouped, attributeCardinality, attributeInGroupCardinality,
      ruleStrengthId, contentTypeId, rangeConstraint, attributeRule

    We primarily need: active, domainId, referencedComponentId (attribute type), rangeConstraint, grouped
    """
    df = read_rf2_txt(
        mrcm_attr_path,
        usecols=["active", "domainId", "referencedComponentId", "rangeConstraint", "grouped"]
    )
    return df[df["active"] == "1"].copy()

def get_mrcm_domain_ids(mrcm_attr_df: pd.DataFrame) -> Set[str]:
    return set(mrcm_attr_df["domainId"].unique())

def resolve_domain_id(concept_id: str, parent_map: Dict[str, Set[str]], domain_ids: Set[str]) -> Optional[str]:
    """
    Return the first ancestor (including self) that is in domain_ids.
    BFS up the hierarchy to find nearest domainId.
    """
    cid = str(concept_id)
    if cid in domain_ids:
        return cid

    visited = set()
    q = deque([cid])
    while q:
        x = q.popleft()
        if x in visited:
            continue
        visited.add(x)
        for p in parent_map.get(x, []):
            if p in domain_ids:
                return p
            q.append(p)
    return None

def allowed_attribute_rows_for_focus(
    focus_concept_id: str,
    mrcm_attr_df: pd.DataFrame,
    parent_map: Dict[str, Set[str]]
) -> pd.DataFrame:
    domain_ids = get_mrcm_domain_ids(mrcm_attr_df)
    dom = resolve_domain_id(focus_concept_id, parent_map, domain_ids)
    if dom is None:
        return mrcm_attr_df.iloc[0:0].copy()  # empty
    return mrcm_attr_df[mrcm_attr_df["domainId"] == dom].copy()

In [58]:
parent_map = load_is_a_parent_map("snomed_source/sct2_Relationship_Snapshot_US1000124_20250901.txt")

In [62]:
att_dom = load_mrcm_attribute_domain("snomed_source/der2_cissccRefset_MRCMAttributeDomainSnapshot_US1000124_20250901.txt")

ValueError: Usecols do not match columns, columns expected but not found: ['rangeConstraint']

In [57]:
import re

ECL_DESC_OR_SELF = re.compile(r'^\s*<<\s*(\d+)\s*$')
ECL_DESC_ONLY    = re.compile(r'^\s*<\s*(\d+)\s*$')

def is_descendant_of(value_id: str, ancestor_id: str, parent_map: Dict[str, Set[str]]) -> bool:
    """
    Check if value_id is a descendant of ancestor_id using parent_map.
    """
    vid = str(value_id)
    anc = str(ancestor_id)
    if vid == anc:
        return True
    visited = set()
    stack = [vid]
    while stack:
        x = stack.pop()
        if x in visited:
            continue
        visited.add(x)
        for p in parent_map.get(x, []):
            if p == anc:
                return True
            stack.append(p)
    return False

def filter_values_by_range_constraint(range_constraint: str, value_ids: List[str], parent_map: Dict[str, Set[str]]) -> List[str]:
    rc = (range_constraint or "").strip()
    if not rc:
        return value_ids  # no constraint given

    m = ECL_DESC_OR_SELF.match(rc)
    if m:
        anc = m.group(1)
        return [v for v in value_ids if is_descendant_of(v, anc, parent_map)]

    m = ECL_DESC_ONLY.match(rc)
    if m:
        anc = m.group(1)
        return [v for v in value_ids if v != anc and is_descendant_of(v, anc, parent_map)]

    # If ECL is complex (AND/OR/exclusions), don't hard-fail; return unfiltered.
    return value_ids

In [ ]:
# ToDo: 
# better explain the similarity !! 
# compare individually, and together
# evaluation is for:
#   1. LLM extraction
#   2. mapping to ontology
#   3. combined if 1 & 2 is correct then it's a correct hit. if either one is incorrect, wrong

##### 2.2.2 Implementation

In [ ]:
python3 mapping_inf.py \
  --pred-jsonl results/contra_ie1/out_gpu0.jsonl results/contra_ie1/out_gpu1.jsonl \
  --out-jsonl results/contra_ie1/mapped_snomed_hits.jsonl


In [1]:
import json
import pandas as pd

In [75]:
output = []
for spl in json_list:
    result = []
    for i,item in enumerate(spl['items']):
        query = item['condition_text']
        hits = search_contraindication(
            query_text=query,
            model=st_model,
            faiss_index=faiss_index,
            meta_df=meta_df,
            es=es,
            bm25_index="snomed_ct_es_index",
            label_column="preferredTerm",
            meta_id_column=None,          # use meta_df.index as id
            bm25_text_field="preferredTerm",
            bm25_id_field="conceptId",
            bm25_label_field=None,
            k_dense=50,
            k_bm25=50,
            k_final=20,
        )
        result.append({
            "item_index": i,
            "query_text": query,
            "hits": hits,
        })
    output.append(result)

In [76]:
output_name = f'results/{current_date}_vaccine_spl_contra_rrf_results.pkl'
with open(output_name, 'wb') as f:
    pickle.dump(output, f)

In [78]:
simple_output = []
for i,spl in enumerate(output):
    for item in spl:
        item_index = item['item_index']
        query_text = item['query_text']
        best_hit_id = item['hits'][0]['id']
        best_hit_label = item['hits'][0]['label']
        simple_output.append({
            "spl_index": i,
            "item_index": item_index,
            "query_text": query_text,
            "best_hit_id": best_hit_id,
            "best_hit_label": best_hit_label,
        })


In [79]:
simple_output

[{'spl_index': 0,
  'item_index': 0,
  'query_text': 'pregnancy',
  'best_hit_id': 77386006,
  'best_hit_label': 'Pregnancy (finding)'},
 {'spl_index': 0,
  'item_index': 1,
  'query_text': 'severe allergic reaction to Adenovirus Type 4 and Type 7 Vaccine, Live, Oral',
  'best_hit_id': 439796003,
  'best_hit_label': 'Live adenovirus type 4 vaccine (substance)'},
 {'spl_index': 0,
  'item_index': 2,
  'query_text': 'inability to swallow tablets whole',
  'best_hit_id': 275929009,
  'best_hit_label': 'Tablets too large to swallow (situation)'},
 {'spl_index': 1,
  'item_index': 0,
  'query_text': 'history of anaphylaxis to BioThrax or vaccine components',
  'best_hit_id': 10839421000119104,
  'best_hit_label': 'History of anaphylaxis (situation)'},
 {'spl_index': 2,
  'item_index': 0,
  'query_text': 'history of severe allergic reaction to CYFENDUS, BioThrax, or vaccine component',
  'best_hit_id': 15920121000119103,
  'best_hit_label': 'Allergic reaction caused by vaccine product (disor

### 3. Phase 03 - Verification

In [82]:
output

[[{'item_index': 0,
   'query_text': 'pregnancy',
   'hits': [{'id': 77386006,
     'label': 'Pregnancy (finding)',
     'fused': 0.03200204813108039,
     'from': 'bm25,dense'},
    {'id': 31601007,
     'label': 'Combined pregnancy (disorder)',
     'fused': 0.03055037313432836,
     'from': 'bm25,dense'},
    {'id': 102874004,
     'label': 'Possible pregnancy (finding)',
     'fused': 0.026655348047538198,
     'from': 'bm25,dense'},
    {'id': 146799005,
     'label': 'Possible pregnancy (situation)',
     'fused': 0.025567754549556326,
     'from': 'bm25,dense'},
    {'id': 11082009,
     'label': 'Abnormal pregnancy (finding)',
     'fused': 0.02546333601933924,
     'from': 'bm25,dense'},
    {'id': 14418008,
     'label': 'Precocious pregnancy (finding)',
     'fused': 0.02353266888150609,
     'from': 'bm25,dense'},
    {'id': 156097009,
     'label': 'Pregnancy complications (disorder)',
     'fused': 0.023042071197411,
     'from': 'bm25,dense'},
    {'id': 87527008,
     '

In [ ]:
# MINIMAL CONTRAINDICATION MAPPING PROMPT (SNOMED CT)
system = """
You are a biomedical terminology assistant that maps contraindication conditions to SNOMED CT concepts.

Your ONLY task is:
- Decide whether any of the provided SNOMED CT candidate concepts is a DIRECT MATCH for the given contraindication text.
- If there is a direct match, select exactly one SNOMED CT concept from the candidate list.
- If there is no direct match, return "N/A".

--------------------
DEFINITIONS
--------------------
A "contraindication condition" is a disease, clinical finding, physiological state, allergy/hypersensitivity, or patient characteristic that makes use of a product inadvisable.

A "DIRECT MATCH" between the query text and a SNOMED CT candidate means:
- They represent the SAME clinical meaning in typical clinical usage, not just overlapping words.
- Minor wording differences or synonyms are acceptable (e.g., "severe combined immunodeficiency" vs. "Severe combined immunodeficiency (disorder)").
- If the candidate is clearly broader or narrower in a clinically important way (e.g., "immunodeficiency" vs. "HIV infection"), it is NOT a direct match unless the query clearly implies that concept.
- Do NOT invent or use any SNOMED CT concept that is not in the candidate list.

--------------------
OUTPUT FORMAT
--------------------
You MUST return ONLY a single JSON object with exactly the following structure:

{{
  "query_text": "<the original contraindication text>",
  "selected_snomed_id": "<the SNOMED CT conceptId if a direct match exists, otherwise 'N/A'>",
  "selected_snomed_term": "<the SNOMED CT preferred term if a direct match exists, otherwise 'N/A'>"
}}

Rules:
- "selected_snomed_id" MUST be either one of the conceptIds from the candidate list OR the string "N/A".
- Do NOT return multiple IDs. Choose the single best direct match or "N/A".
- Do NOT include any explanations, reasoning, comments, or extra fields.
- Do NOT change the JSON keys or structure.

--------------------
TASK
--------------------
You will be given:
- A "QUERY" containing the contraindication text to be normalized.
- A "CANDIDATES" list that contains possible SNOMED CT concepts.

Carefully compare the QUERY with the CANDIDATES and output ONLY the JSON object described above. Do not show your reasoning. Provide only the final JSON answer.
"""

user = """
QUERY:
"{query_text}"

CANDIDATES (each line is one SNOMED CT concept):
{hits_json}

"""


In [ ]:
system = """
You are a biomedical terminology assistant that maps contraindication conditions to SNOMED CT concepts.

Your ONLY task:
- Decide whether any provided candidate is a DIRECT MATCH for the query.
- If a direct match exists, select exactly ONE candidate concept.
- Otherwise return "N/A".
You MUST NOT use any concept outside the candidate list.

--------------------
DIRECT MATCH (STRICT)
--------------------
A candidate is a DIRECT MATCH only if it represents the SAME clinical meaning as the query in typical clinical usage.

A direct match MUST preserve:
1) Clinical concept identity (same condition/finding/state)
2) Polarity/temporality:
   - "history of X" ≠ "X"
   - "suspected X" ≠ "X"
3) Allergy/hypersensitivity target:
   - "hypersensitivity to drug A" ≠ "hypersensitivity to ingredients" ≠ "hypersensitivity to drug class"
4) Appropriate semantic type:
   - If the query is a CONDITION/FINDING, do NOT select PROCEDURE/PRODUCT concepts.

If no candidate satisfies ALL applicable constraints above, return "N/A".

--------------------
CANDIDATE ELIGIBILITY FILTERS
--------------------
Unless the query explicitly describes a procedure or product, DO NOT select candidates whose label indicates:
- procedure (e.g., "administration", "vaccination", "(procedure)")
- product or substance concepts when query is a condition (unless query itself is "allergy to <substance>")
- overly broad parent concepts when query is specific

--------------------
OUTPUT FORMAT (STRICT)
--------------------
Return ONLY a single line of MINIFIED JSON followed immediately by the token <<END_JSON>>.
Do NOT use markdown fences (no ```).
Do NOT include explanations or extra fields.

Exact structure:

{"query_text":"<original query>","selected_snomed_id":"<candidate id or N/A>","selected_snomed_term":"<candidate label or N/A>"}<<END_JSON>>

Rules:
- selected_snomed_id must be exactly one candidate "id" value OR "N/A".
- selected_snomed_term must be the corresponding candidate "label" OR "N/A".
- Choose ONE best direct match or "N/A". Never output multiple ids.
"""

user = """
QUERY:
"{query_text}"

CANDIDATES (each line is one SNOMED CT concept):
{hits_json}

"""
# (JSON array; each object has keys: id, label):


##### testing

In [102]:
def flatten(data):
    return [item for sub in data for item in sub]


In [121]:
import json
from typing import List, Dict, Any

def add_hits_json(records: List[Dict[str, Any]], output: List[str]) -> List[Dict[str, Any]]:
    """
    Add a JSON-serialized 'hits_json' field to each record, containing
    only the specified keys from each hit.

    Example:
      output = ["id", "label"]
      hits_json -> [{"id": 123, "label": "Pregnancy (finding)"}, ...]
    """
    payloads = []
    for r in records:
        hits = r.get("hits", []) or []

        # Keep only requested keys per hit
        filtered_hits = [
            {k: h.get(k) for k in output if k in h}
            for h in hits
        ]

        payloads.append({
            **r,
            "hits_json": json.dumps(
                filtered_hits,
                indent=2,
                ensure_ascii=False
            )
        })
    return payloads


In [138]:
records = flatten(output)
payloads = add_hits_json(records, output=['id','label'])

In [139]:
# sections = rx_spl_df['contra_text'].tolist()[:50]
msgs = build_messages_from_iter(system, user, payloads[:10])

In [130]:
model = load_model_local("google/medgemma-27b-text-it")

Loading checkpoint shards:   0%|          | 0/11 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [141]:
out = [model.generate(m, max_new_tokens=512) for m in msgs[:2]]

skipping cudagraphs due to skipping cudagraphs due to multiple devices: device(type='cuda', index=0), device(type='cuda', index=1)
skipping cudagraphs due to skipping cudagraphs due to multiple devices: device(type='cuda', index=0), device(type='cuda', index=1)


In [142]:
out[0]

'```json\n{\n  "query_text": "pregnancy",\n  "selected_snomed_id": "77386006",\n  "selected_snomed_term": "Pregnancy (finding)"\n}\n```thought\nThe user wants me to find the best SNOMED CT match for the query "pregnancy" from the provided list of candidates.\n\nThe query is simply "pregnancy".\n\nLooking at the candidates:\n- "Pregnancy (finding)" (77386006) seems like a very direct match. It represents the state of being pregnant.\n- "Combined pregnancy (disorder)" (31601007) is too specific (multiple fetuses).\n- "Possible pregnancy (finding)" (102874004) and "Possible pregnancy (situation)" (146799005) are too uncertain.\n- "Abnormal pregnancy (finding)" (11082009) is too specific.\n- "Precocious pregnancy (finding)" (14418008) is too specific.\n- "Pregnancy complications (disorder)" (156097009) is too specific.\n- "Term pregnancy (finding)" (87527008) is too specific.\n- "Prolonged pregnancy (disorder)" (90968009, 156122006) is too specific.\n- "Abdominal pregnancy (disorder)" (826

In [143]:
out[1]

'```json\n{\n  "query_text": "severe allergic reaction to Adenovirus Type 4 and Type 7 Vaccine, Live, Oral",\n  "selected_snomed_id": "N/A",\n  "selected_snomed_term": "N/A"\n}\n```thought\nThe user wants me to find a direct match for the contraindication "severe allergic reaction to Adenovirus Type 4 and Type 7 Vaccine, Live, Oral" within the provided list of SNOMED CT concepts.\n\nThe query describes a specific type of adverse reaction (severe allergic reaction) to a specific vaccine (Adenovirus Type 4 and Type 7 Vaccine, Live, Oral).\n\nLet\'s examine the candidates:\n- Many candidates relate to Adenovirus vaccines (types 4 and 7), but they are substances or products, not the reaction itself.\n- Some candidates relate to Adenovirus infection or antigens, which are different from a reaction to a vaccine.\n- Some candidates are for canine vaccines or other unrelated conditions/procedures.\n- None of the candidates directly represent "severe allergic reaction to Adenovirus Type 4 and T

In [144]:
out[2]

IndexError: list index out of range

In [ ]:
# Run and evluate on a random selection including Vaccine + drugs 
# ToDo: # check on # of SPLs and then come up with a number !! 

In [ ]:
# read picked file
import pickle
out_name = "results/2025-12-16_vaccine_spl_contra_verified.pkl"
with open(out_name, 'rb') as f:
    verified = pickle.load(f)

In [6]:
# read jsonl file into dataframe 
res_json= 'results/contra_ie2/verified_hits.jsonl'
res_df = pd.read_json(res_json, lines=True)
res_df.to_csv('results/contra_ie2/verified_hits.csv')

{'SPL_SET_ID': 'cb8e8155-5505-44d5-9dee-525e2e45b95d',
 'item_index': 0,
 'query_text': 'hypersensitivity to pentamidine isethionate',
 'selected_snomed_id': 'N/A',
 'selected_snomed_term': 'N/A'}

In [5]:
verified[0]

{'query_text': 'pregnancy',
 'selected_snomed_id': '77386006',
 'selected_snomed_term': 'Pregnancy (finding)'}

In [9]:
verify_df = pd.DataFrame(verified)
verify_df

,query_text,selected_snomed_id,selected_snomed_term
0,pregnancy,77386006,Pregnancy (finding)
1,severe allergic reaction to Adenovirus Type 4 ...,N/A,N/A
2,inability to swallow tablets whole,702551003,Unable to swallow tablet (finding)
3,history of anaphylaxis to BioThrax or vaccine ...,161595008,History of vaccine allergy (situation)
4,history of severe allergic reaction to CYFENDU...,161595008,History of vaccine allergy (situation)
...,...,...,...
207,history of severe allergic reaction to previou...,735933002,History of allergy to drug (situation)
208,history of severe allergic reaction to vaccine...,315789000,[V]Personal history of serum or vaccine allerg...
209,history of severe allergic reaction to previou...,735933002,History of allergy to drug (situation)
210,history of severe allergic reaction to vaccine...,315789000,[V]Personal history of serum or vaccine allerg...


### Post-Coordination [AGENTIC]

#### Hypersensitivity Decomposer - Baseline (NON-LLM)

In [1]:
import re
from typing import Any, Dict, List, Optional, Tuple

# --- Constants for the postcoord pattern (SNOMED CT) ---
PROPENSITY_AR = {"id": 420134006, "label": "Propensity to adverse reaction (finding)"}
HAS_REALIZATION = {"id": 719722006, "label": "Has realization (attribute)"}
HYPERSENS_PROCESS = {"id": 472963003, "label": "Hypersensitivity process (qualifier value)"}
CAUSATIVE_AGENT = {"id": 246075003, "label": "Causative agent (attribute)"}

# --- Regexes to detect and extract the agent ---
RX_HS_TO = re.compile(
    r"\b(?:known\s+)?(?:hypersensitivity|allergy|anaphylaxis|allergic\s+reaction)\s+(?:to|against)\s+(?P<agent>.+)$",
    re.IGNORECASE,
)
RX_AGENT_HS = re.compile(
    r"^(?P<agent>.+?)\s+(?:hypersensitivity|allergy)\b",
    re.IGNORECASE,
)

# Basic stopwords/cleanup for agent mentions
_AGENT_CLEAN = [
    r"^\b(a|an|the)\b\s+",
    r"^\bany\s+of\s+the\b\s+",
    r"^\bany\s+of\b\s+",
    r"^\bother\b\s+",
    r"^\bknown\b\s+",
    r"^\bwith\b\s+",
]

def _norm_space(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    return s

def is_hypersensitivity_statement(text: str) -> bool:
    t = (text or "").lower()
    return any(k in t for k in ["hypersensitivity", "allergy", "anaphylaxis", "allergic reaction"])

def extract_hypersensitivity_agent(query_text: str) -> Optional[str]:
    """
    Extract the agent phrase from a hypersensitivity/allergy statement.
    Returns agent text (string) or None.
    """
    q = _norm_space(query_text)

    m = RX_HS_TO.search(q)
    if m:
        agent = m.group("agent")
    else:
        m2 = RX_AGENT_HS.match(q)
        if not m2:
            return None
        agent = m2.group("agent")

    agent = _norm_space(agent)
    # Trim trailing punctuation
    agent = agent.rstrip(" .;:,")

    # Heuristic: if the statement is "hypersensitivity to X or Y", caller should split earlier
    # Here we just keep as-is.

    # Cleanup leading boilerplate tokens
    a = agent
    for pat in _AGENT_CLEAN:
        a = re.sub(pat, "", a, flags=re.IGNORECASE)
    a = _norm_space(a)
    return a or None

def _looks_like_substance_label(label: str) -> bool:
    # RF2 labels often include "(substance)" suffix; candidates might also include it.
    return "(substance)" in (label or "").lower()

def _looks_like_hs_finding_label(label: str) -> bool:
    # "Hypersensitivity to X (finding)" and similar
    l = (label or "").lower()
    return "hypersensitivity" in l and "(finding)" in l

def choose_best_substance_candidate(agent_text: str, hits: List[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    """
    Select a SNOMED substance candidate from hits using simple lexical scoring.
    hits: list of dicts with at least {"id":..., "label":...}
    Returns best hit dict or None.
    """
    agent = (agent_text or "").lower().strip()
    if not agent:
        return None

    subs = [h for h in hits if _looks_like_substance_label(h.get("label", ""))]
    if not subs:
        return None

    # Very simple scoring: exact substring match gets priority
    def score(h: Dict[str, Any]) -> Tuple[int, int]:
        label = (h.get("label") or "").lower()
        # remove semantic tag for matching
        label_clean = re.sub(r"\s*\(substance\)\s*$", "", label).strip()
        exact = 1 if (label_clean == agent) else 0
        contains = 1 if agent and agent in label_clean else 0
        # longer overlap heuristic
        overlap = len(set(agent.split()) & set(label_clean.split()))
        return (exact * 100 + contains * 10 + overlap, -len(label_clean))

    subs_sorted = sorted(subs, key=score, reverse=True)
    return subs_sorted[0] if subs_sorted else None

def hypersensitivity_decompose(
    query_text: str,
    hits: List[Dict[str, Any]],
) -> Dict[str, Any]:
    """
    Decompose a hypersensitivity contraindication into either:
    - direct precoord concept (if present), or
    - postcoord expression Propensity to adverse reaction with realization Hypersensitivity process
      and causative agent = mapped substance.

    Returns a structured object (always).
    """
    qt = _norm_space(query_text)

    # 0) If not hypersensitivity-like, return a "not_applicable" result
    if not is_hypersensitivity_statement(qt):
        return {
            "query_text": qt,
            "template": "NOT_HYPERSENSITIVITY",
            "direct_match": {"selected_snomed_id": "N/A", "selected_snomed_term": "N/A"},
            "postcoord_expression": "N/A",
            "components": {},
        }

    # 1) Prefer direct pre-coordinated hypersensitivity finding if present in candidates
    hs_findings = [h for h in hits if _looks_like_hs_finding_label(h.get("label", ""))]
    if hs_findings:
        # Pick best by lexical similarity to query_text
        ql = qt.lower()
        def fscore(h: Dict[str, Any]) -> Tuple[int, int]:
            lbl = (h.get("label") or "").lower()
            contains = 1 if ("hypersensitivity" in lbl and any(tok in lbl for tok in ql.split())) else 0
            return (contains, -len(lbl))
        hs_best = sorted(hs_findings, key=fscore, reverse=True)[0]
        return {
            "query_text": qt,
            "template": "T1_HYPERSENSITIVITY_DIRECT",
            "direct_match": {
                "selected_snomed_id": str(hs_best["id"]),
                "selected_snomed_term": hs_best.get("label", ""),
            },
            "postcoord_expression": "N/A",
            "components": {},
        }

    # 2) Otherwise build postcoord from components
    agent_text = extract_hypersensitivity_agent(qt)

    # Edge: "hypersensitivity to ingredients/components of X"
    # For now we still try to map "X" (or the remaining phrase) to a substance; if it fails, return N/A agent.
    agent_hit = choose_best_substance_candidate(agent_text or "", hits)

    if not agent_hit:
        return {
            "query_text": qt,
            "template": "T1_HYPERSENSITIVITY_POSTCOORD",
            "direct_match": {"selected_snomed_id": "N/A", "selected_snomed_term": "N/A"},
            "postcoord_expression": "N/A",
            "components": {
                "base_finding": PROPENSITY_AR,
                "has_realization": HAS_REALIZATION,
                "realization_value": HYPERSENS_PROCESS,
                "causative_agent": CAUSATIVE_AGENT,
                "agent_text": agent_text or "N/A",
                "agent_substance": {"id": "N/A", "label": "N/A"},
            },
            "note": "No SNOMED substance candidate could be selected for causative agent.",
        }

    expr = f"{PROPENSITY_AR['id']}:" \
           f"{{{HAS_REALIZATION['id']}={HYPERSENS_PROCESS['id']}," \
           f"{CAUSATIVE_AGENT['id']}={agent_hit['id']}}}"

    return {
        "query_text": qt,
        "template": "T1_HYPERSENSITIVITY_POSTCOORD",
        "direct_match": {"selected_snomed_id": "N/A", "selected_snomed_term": "N/A"},
        "postcoord_expression": expr,
        "components": {
            "base_finding": PROPENSITY_AR,
            "has_realization": HAS_REALIZATION,
            "realization_value": HYPERSENS_PROCESS,
            "causative_agent": CAUSATIVE_AGENT,
            "agent_text": agent_text or "N/A",
            "agent_substance": {"id": agent_hit["id"], "label": agent_hit.get("label", "")},
        },
    }


In [2]:
query = "known hypersensitivity to oxymorphone"
hits = [
    {"id": 24751001, "label": "Oxymorphone (substance)"},
    {"id": 472963003, "label": "Hypersensitivity process (qualifier value)"},  # might appear; ignored
]
print(hypersensitivity_decompose(query, hits))


{'query_text': 'known hypersensitivity to oxymorphone', 'template': 'T1_HYPERSENSITIVITY_POSTCOORD', 'direct_match': {'selected_snomed_id': 'N/A', 'selected_snomed_term': 'N/A'}, 'postcoord_expression': '420134006:{719722006=472963003,246075003=24751001}', 'components': {'base_finding': {'id': 420134006, 'label': 'Propensity to adverse reaction (finding)'}, 'has_realization': {'id': 719722006, 'label': 'Has realization (attribute)'}, 'realization_value': {'id': 472963003, 'label': 'Hypersensitivity process (qualifier value)'}, 'causative_agent': {'id': 246075003, 'label': 'Causative agent (attribute)'}, 'agent_text': 'oxymorphone', 'agent_substance': {'id': 24751001, 'label': 'Oxymorphone (substance)'}}}


# Contraindication patterns: 
* Disorders (T0) - Direct mappings
* Hypersensitivity complex (T1)
* Coadministrations (T2)
* Combinations (T3) (with different qualifiers like temporal or population)

# Tools for Agent: 
1. candidate generator --> from VaxMapper.src.utils.search_utils import search_query [DONE]
2. Definition retriever --> RAG tool to search top-k similar concepts + definitions
3. Post-Coord Builder --> Builds expression from learned/similar concept's definitions
4. 



### T1 - Hypersensitivity to {Agent}

In [ ]:
# from collections import defaultdict

# def generate_scg_from_table(relationships):
# 	"""
# 	Generates a SNOMED CT Compositional Grammar (SCG) expression 
# 	from a list of stated relationship rows.
# 	"""
# 	parents = []
# 	ungrouped_attributes = []
# 	grouped_attributes = defaultdict(list)

# 	# 1. Parse the table rows
# 	for rel in relationships:
# 		# Standard SNOMED codes
# 		IS_A = "116680003"

# 		type_id = str(rel['typeId'])
# 		type_label = rel.get('typeLabel', type_id)
# 		target_id = str(rel['destinationId'])
# 		target_label = rel.get('destinationLabel', target_id)
# 		group_id = int(rel['relationshipGroup'])

# 		formatted_pair = f"{type_id} |{type_label}| = {target_id} |{target_label}|"

# 		# 2. Categorize by Is-A, Ungrouped (0), or Grouped (>0)
# 		if type_id == IS_A:
# 			parents.append(f"{target_id} |{target_label}|")
# 		elif group_id == 0:
# 			ungrouped_attributes.append(formatted_pair)
# 		else:
# 			grouped_attributes[group_id].append(formatted_pair)

# 	# 3. Assemble the SCG segments
# 	# Parents are joined by '+'
# 	expression = " + ".join(parents)

# 	# Build refinement sections
# 	refinements = []
	
# 	# Add ungrouped attributes
# 	if ungrouped_attributes:
# 		refinements.extend(ungrouped_attributes)
	
# 	# Add grouped attributes enclosed in {}
# 	for group_num in sorted(grouped_attributes.keys()):
# 		group_content = ", ".join(grouped_attributes[group_num])
# 		refinements.append(f"{{{group_content}}}")

# 	# 4. Final Concatenation
# 	if refinements:
# 		expression += " : " + ", ".join(refinements)

# 	return expression

In [ ]:
# import re

# def _strip_semantic_tag(term):
#     if pd.isna(term):
#         return None
#     return re.sub(r"\s*\([^)]*\)$", "", str(term)).strip()

# def extract_snomed_relationships(sourceId, snomed_rel_df=snomed_rel_df, concept_df=concept_df):
#     """
#     Return SNOMED relationships for a source concept with readable labels.

#     Output shape per row:
#     {
#         "typeId": "116680003",
#         "typeLabel": "Is a",
#         "destinationId": "80146002",
#         "destinationLabel": "Appendectomy",
#         "relationshipGroup": 0
#     }
#     """
#     source_id = int(sourceId)

#     rel_cols = ["typeId", "destinationId", "relationshipGroup"]
#     out_cols = ["typeId", "typeLabel", "destinationId", "destinationLabel", "relationshipGroup"]

#     if source_id not in snomed_rel_df.index:
#         return []

#     rel_df = snomed_rel_df.loc[[source_id], rel_cols].copy()
#     rel_df = rel_df.reset_index(drop=True)

#     concept_map = (
#         concept_df[["conceptId", "term"]]
#         .drop_duplicates(subset=["conceptId"])
#         .set_index("conceptId")["term"]
#     )

#     rel_df["typeLabel"] = rel_df["typeId"].map(concept_map).apply(_strip_semantic_tag)
#     rel_df["destinationLabel"] = rel_df["destinationId"].map(concept_map).apply(_strip_semantic_tag)

#     rel_df["typeId"] = rel_df["typeId"].astype(str)
#     rel_df["destinationId"] = rel_df["destinationId"].astype(str)
#     rel_df["relationshipGroup"] = rel_df["relationshipGroup"].astype(int)

#     return rel_df[out_cols].to_dict(orient="records")

# # Example:
# # extract_snomed_relationships(174041007)


In [ ]:
# # --- Example Usage ---
# # Simulating a "Stated Relationship Table" for: 
# # 174041007 |Laparoscopic emergency appendectomy|
# stated_table = [
#     {"typeId": "116680003", "typeLabel": "Is a", "destinationId": "80146002", "destinationLabel": "Appendectomy", "relationshipGroup": 0},
#     {"typeId": "116680003", "typeLabel": "Is a", "destinationId": "80146002", "destinationLabel": "Appendectomy", "relationshipGroup": 0},
#     {"typeId": "405813007", "typeLabel": "Procedure site - Direct", "destinationId": "66754008", "destinationLabel": "Appendix structure", "relationshipGroup": 1},
#     {"typeId": "260870009", "typeLabel": "Priority", "destinationId": "25876001", "destinationLabel": "Emergency", "relationshipGroup": 1},
#     {"typeId": "425392003", "typeLabel": "Using access device", "destinationId": "86174004", "destinationLabel": "Laparoscope", "relationshipGroup": 1}
# ]

# scg_expression = generate_scg_from_table(stated_table)
# print(f"Generated SCG:\n{scg_expression}")

Generated SCG:
80146002 |Appendectomy| + 80146002 |Appendectomy| : {405813007 |Procedure site - Direct| = 66754008 |Appendix structure|, 260870009 |Priority| = 25876001 |Emergency|, 425392003 |Using access device| = 86174004 |Laparoscope|}


In [ ]:
# get active/inactive ingredients from SPL document
def retrieve_ingredients(set_id):

In [24]:
from VaxMapper.src.utils.snomed_utils import generate_scg_from_table, extract_snomed_relationships

In [27]:
snomed_rel_df = pd.read_csv('snomed_us_source/sct2_Relationship_Snapshot_US1000124_20250901.txt', sep='\t')
snomed_rel_df = snomed_rel_df[(snomed_rel_df['active']==1) ] # ACTIVE  DEFINITIONS ONLY
snomed_rel_df = snomed_rel_df[['active','sourceId', 'destinationId', 'relationshipGroup', 'typeId','characteristicTypeId']].set_index('sourceId')
snomed_rel_df

,active,destinationId,relationshipGroup,typeId,characteristicTypeId
sourceId,,,,,
10000006,1,29857009,0,116680003,900000000000011006
10000006,1,9972008,0,116680003,900000000000011006
134035007,1,84371003,0,116680003,900000000000011006
134136005,1,57250008,0,116680003,900000000000011006
10002003,1,116175006,0,116680003,900000000000011006
...,...,...,...,...,...
900000000000546006,1,900000000000491004,0,116680003,900000000000011006
900000000000547002,1,900000000000480006,0,116680003,900000000000011006
900000000000548007,1,900000000000511003,0,116680003,900000000000011006


In [25]:
out = extract_snomed_relationships(1269354002,snomed_rel_df,concept_df)
out

[{'typeId': '116680003',
  'typeLabel': 'Is a',
  'destinationId': '419511003',
  'destinationLabel': 'Propensity to adverse reactions to drug',
  'relationshipGroup': 0},
 {'typeId': '116680003',
  'typeLabel': 'Is a',
  'destinationId': '609433001',
  'destinationLabel': 'Hypersensitivity disposition',
  'relationshipGroup': 0},
 {'typeId': '719722006',
  'typeLabel': 'Has realization',
  'destinationId': '472963003',
  'destinationLabel': 'Hypersensitivity process',
  'relationshipGroup': 1},
 {'typeId': '246075003',
  'typeLabel': 'Causative agent',
  'destinationId': '372694001',
  'destinationLabel': 'Erythromycin',
  'relationshipGroup': 1}]

In [26]:
x = generate_scg_from_table(out)
x[0]

'419511003 |Propensity to adverse reactions to drug| + 609433001 |Hypersensitivity disposition| : {719722006 |Has realization| = 472963003 |Hypersensitivity process|, 246075003 |Causative agent| = 372694001 |Erythromycin|}'

In [17]:
out

[{'typeId': '116680003',
  'typeLabel': 'Is a',
  'destinationId': '419511003',
  'destinationLabel': 'Propensity to adverse reactions to drug',
  'relationshipGroup': 0},
 {'typeId': '116680003',
  'typeLabel': 'Is a',
  'destinationId': '609433001',
  'destinationLabel': 'Hypersensitivity disposition',
  'relationshipGroup': 0},
 {'typeId': '719722006',
  'typeLabel': 'Has realization',
  'destinationId': '472963003',
  'destinationLabel': 'Hypersensitivity process',
  'relationshipGroup': 1},
 {'typeId': '246075003',
  'typeLabel': 'Causative agent',
  'destinationId': '372694001',
  'destinationLabel': 'Erythromycin',
  'relationshipGroup': 1}]

In [ ]:
system = """
You are a SNOMED CT modeling assistant.

You will be given:
- A query contraindication
- Several similar example SNOMED CT concepts with their INFERRED relationship axioms grouped by role group and their respective SNOMED CT Compositional Grammar (SCG) expression. 

TASK:
1) Identify the COMMON modeling pattern used by the examples for hypersensitivity to an agent.
2) Output a JSON object that contains:
   - pattern_attributes: the repeated attribute typeIds and their repeated destinationIds (if constant)
   - variable_slots: which attribute value is the agent (variable)
   - proposed_expression_template: a SNOMED compositional grammar template
3) If the examples do not support a consistent pattern, output pattern_found=false.

--------------------
OUTPUT FORMAT 
--------------------
OUTPUT ONLY MINIFIED JSON followed by <<END_JSON>>.

Exact structure:
{{"pattern_found: "True or False", expression:"<proposed expression>"}}<<END_JSON>>

"""
user = """
QUERY:
"{query_text}"

CANDIDATES (each line is one SNOMED CT concept definition and SNOMED CT Compositional Grammar (SCG) expression):
{hits_json}

"""



In [27]:
## Issue with lexical/semantic similarity. (structural similarity to be added)

from typing import Dict, Any, List, Tuple, Set
from collections import Counter

ISA = "116680003"
STATED = "900000000000010007"

def concept_signature_from_rels(rels: List[Dict[str, Any]]) -> Dict[str, Set[str]]:
    """
    rels format: list of relationship dicts (e.g., output from extract_snomed_relationships),
    where each row has at least: typeId, destinationId, relationshipGroup.

    Returns:
      - type_only: set of "g:<group>|t:<typeId>"
      - type_value: set of "g:<group>|t:<typeId>|v:<destId>"
    """
    type_only: Set[str] = set()
    type_value: Set[str] = set()

    for rel in rels or []:
        t = str(rel.get("typeId", ""))
        if not t or t == ISA:
            continue

        grp = int(rel.get("relationshipGroup", 0))
        v = str(rel.get("destinationId", ""))
        if not v:
            continue

        type_only.add(f"g:{grp}|t:{t}")
        type_value.add(f"g:{grp}|t:{t}|v:{v}")

    return {"type_only": type_only, "type_value": type_value}

In [28]:
concept_signature_from_rels(out)

{'type_only': {'g:1|t:246075003', 'g:1|t:719722006'},
 'type_value': {'g:1|t:246075003|v:372694001', 'g:1|t:719722006|v:472963003'}}

In [29]:
import math

def weighted_jaccard(a: Set[str], b: Set[str], idf: Dict[str, float]) -> float:
    if not a and not b:
        return 0.0
    inter = a & b
    union = a | b
    num = sum(idf.get(x, 1.0) for x in inter)
    den = sum(idf.get(x, 1.0) for x in union)
    return float(num / den) if den else 0.0

In [30]:
def build_idf(signatures: List[Set[str]]) -> Dict[str, float]:
    N = len(signatures)
    df = Counter()
    for s in signatures:
        for feat in s:
            df[feat] += 1
    return {feat: math.log((N + 1) / (cnt + 1)) for feat, cnt in df.items()}

In [32]:
def select_structured_exemplars(
    candidates: List[Any],
    concept_df,
    id_col: str = "conceptId",
    term_col: str = "term",
    top_k: int = 8,
    min_attrs: int = 2
) -> List[Dict[str, Any]]:
    """
    candidates: [{id,label,...}, ...] or [conceptId, ...]
    concept_df: dataframe used to resolve labels from id_col -> term_col
    Strategy:
      1) compute type_only signatures for each
      2) compute IDF weights within candidate pool
      3) score each candidate by average similarity to others (cohesion)
      4) return most 'central' structured concepts
    """
    term_lookup = (
        concept_df[[id_col, term_col]]
        .dropna(subset=[id_col])
        .drop_duplicates(subset=[id_col])
        .assign(**{id_col: lambda d: d[id_col].astype(str)})
        .set_index(id_col)[term_col]
        .to_dict()
    )

    # build signatures
    sigs = []
    kept = []
    for c in candidates:
        h = c if isinstance(c, dict) else {"id": c}
        cid = h.get("id")
        if cid is None:
            continue
        cid_str = str(cid)
        label = h.get("label")
        if label is None or str(label).strip() == "":
            label = term_lookup.get(cid_str, cid_str)
        h = {**h, "id": cid_str, "label": label}

        rels = extract_snomed_relationships(cid, snomed_rel_df,concept_df)
        sig = concept_signature_from_rels(rels)["type_only"]
        if len(sig) < min_attrs:
            continue
        kept.append(h)
        sigs.append(sig)

    if not kept:
        return []

    idf = build_idf(sigs)

    # cohesion score: average similarity to others
    scores = []
    for i, si in enumerate(sigs):
        sim_sum = 0.0
        denom = 0
        for j, sj in enumerate(sigs):
            if i == j:
                continue
            sim_sum += weighted_jaccard(si, sj, idf)
            denom += 1
        scores.append(sim_sum / denom if denom else 0.0)

    ranked = sorted(zip(kept, scores), key=lambda x: x[1], reverse=True)
    return [h for h, _ in ranked[:top_k]]

In [33]:
candidates = [24751001,69899006,121420005,370266008,1269395004,777032000,1269398002,421961002,1269354002,1269400003,252097006,473010000,609532009,609433001,609398007,703076003,29268000,1163444007,780099002,1269352003]
select_structured_exemplars(candidates, concept_df=concept_df, id_col="conceptId", term_col="term")

[{'id': '1269395004', 'label': 'Hypersensitivity to hydrocortisone (finding)'},
 {'id': '1269398002', 'label': 'Hypersensitivity to minocycline (finding)'},
 {'id': '1269354002', 'label': 'Hypersensitivity to erythromycin (finding)'},
 {'id': '1269400003', 'label': 'Hypersensitivity to nalidixic acid (finding)'},
 {'id': '609532009',
  'label': 'Non-allergic hypersensitivity to sodium thiosulfate (finding)'},
 {'id': '609398007',
  'label': 'Non-allergic hypersensitivity to drug or medicament (finding)'},
 {'id': '703076003',
  'label': 'Non-allergic hypersensitivity to benzoic acid (finding)'},
 {'id': '1163444007', 'label': 'Hypersensitivity to macrogol (finding)'}]

In [35]:
candidates2=[195967001,281239006,733858005,708090002,734905008,762521001,708094006,425969006,707445000,708096008,708093000,707980005,707979007,707981009,782513000,734904007,707446004,10692721000119102,10675311000119105,10674791000119101]
select_structured_exemplars(candidates2, concept_df=concept_df, id_col="conceptId", term_col="term")

[{'id': '281239006', 'label': 'Exacerbation of asthma (disorder)'},
 {'id': '708094006',
  'label': 'Acute exacerbation of intrinsic asthma (disorder)'},
 {'id': '425969006',
  'label': 'Exacerbation of intermittent asthma (disorder)'},
 {'id': '707445000',
  'label': 'Exacerbation of mild persistent asthma (disorder)'},
 {'id': '708096008',
  'label': 'Acute severe exacerbation of intrinsic asthma (disorder)'},
 {'id': '707980005',
  'label': 'Acute severe exacerbation of moderate persistent asthma (disorder)'},
 {'id': '707979007',
  'label': 'Acute severe exacerbation of severe persistent asthma (disorder)'},
 {'id': '707981009',
  'label': 'Acute severe exacerbation of mild persistent asthma (disorder)'}]

In [ ]:
## Patch: Use most common qualifier values and their codes as a dictionary. Lookup and decompose. 
# Better way would be to embed/bm25 all qualifier values in the hierarchy and do the lookup

In [82]:
from pathlib import Path
from typing import Any, Dict, Iterable, List
import json

def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def iter_payloads(mapped_rows: Iterable[Dict[str, Any]]) -> List[Dict[str, Any]]:
    payloads: List[Dict[str, Any]] = []
    for row in mapped_rows:
        spl_set_id = str(row.get("SPL_SET_ID", ""))
        items = row.get("items") or []
        for item in items:
            hits = item.get("hits") or []
            filtered_hits = [{"id": h.get("id"), "label": h.get("label")} for h in hits]
            payloads.append(
                {
                    "SPL_SET_ID": spl_set_id,
                    "item_index": item.get("item_index"),
                    "query_text": str(item.get("query_text", "")),
                    "hits": filtered_hits,
                    "hits_json": json.dumps(filtered_hits, ensure_ascii=False, indent=2),
                }
            )
    return payloads


def build_hit_relationship_scg_payloads(mapped_rows: Iterable[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Build one payload per hit from mapped JSONL rows.

    Each payload includes:
      1) query_text
      2) relationships from extract_snomed_relationships(hit_id)
      3) scg_expression from generate_scg_from_table(relationships)
    """
    payloads: List[Dict[str, Any]] = []
    for row in mapped_rows:
        spl_set_id = str(row.get("SPL_SET_ID", ""))
        items = row.get("items") or []
        for item in items:
            query_text = str(item.get("query_text", ""))
            item_index = item.get("item_index")
            hits = item.get("hits") or []

            for rank, hit in enumerate(hits):
                hit_id = hit.get("id")
                hit_label = hit.get("label")

                try:
                    relationships = extract_snomed_relationships(hit_id)
                    scg_expression = generate_scg_from_table(relationships) if relationships else ""
                except Exception:
                    relationships = []
                    scg_expression = ""

                payloads.append(
                    {
                        "SPL_SET_ID": spl_set_id,
                        "item_index": item_index,
                        "query_text": query_text,
                        "hit_rank": rank,
                        "hit_id": str(hit_id) if hit_id is not None else "",
                        "hit_label": hit_label,
                        "relationships": relationships,
                        "scg_expression": scg_expression,
                    }
                )

    return payloads

In [76]:
test = 'results/contra_ie2/mapped_hits.jsonl'
mapped_rows = read_jsonl(test)
all_payloads = iter_payloads(mapped_rows)

In [83]:
mapped_rows = read_jsonl("results/contra_ie2/mapped_hits.jsonl")
hit_payloads = build_hit_relationship_scg_payloads(mapped_rows)
hit_payloads[0]


{'SPL_SET_ID': '285e60f8-2404-47d0-a171-a4f075d6f418',
 'item_index': 0,
 'query_text': 'hypersensitivity to oxymorphone',
 'hit_rank': 0,
 'hit_id': '24751001',
 'hit_label': 'Oxymorphone (substance)',
 'relationships': [{'typeId': '116680003',
   'typeLabel': 'Is a',
   'destinationId': '404642006',
   'destinationLabel': 'Substance with opioid receptor agonist mechanism of action',
   'relationshipGroup': 0},
  {'typeId': '726542003',
   'typeLabel': 'Has disposition',
   'destinationId': '734727006',
   'destinationLabel': 'Opioid receptor agonist',
   'relationshipGroup': 0},
  {'typeId': '116680003',
   'typeLabel': 'Is a',
   'destinationId': '766763002',
   'destinationLabel': 'Morphinane alkaloid',
   'relationshipGroup': 0},
  {'typeId': '116680003',
   'typeLabel': 'Is a',
   'destinationId': '373265006',
   'destinationLabel': 'Analgesic',
   'relationshipGroup': 0},
  {'typeId': '116680003',
   'typeLabel': 'Is a',
   'destinationId': '9680003',
   'destinationLabel': 'Cen

In [85]:
hit_payloads[0]

{'SPL_SET_ID': '285e60f8-2404-47d0-a171-a4f075d6f418',
 'item_index': 0,
 'query_text': 'hypersensitivity to oxymorphone',
 'hit_rank': 0,
 'hit_id': '24751001',
 'hit_label': 'Oxymorphone (substance)',
 'relationships': [{'typeId': '116680003',
   'typeLabel': 'Is a',
   'destinationId': '404642006',
   'destinationLabel': 'Substance with opioid receptor agonist mechanism of action',
   'relationshipGroup': 0},
  {'typeId': '726542003',
   'typeLabel': 'Has disposition',
   'destinationId': '734727006',
   'destinationLabel': 'Opioid receptor agonist',
   'relationshipGroup': 0},
  {'typeId': '116680003',
   'typeLabel': 'Is a',
   'destinationId': '766763002',
   'destinationLabel': 'Morphinane alkaloid',
   'relationshipGroup': 0},
  {'typeId': '116680003',
   'typeLabel': 'Is a',
   'destinationId': '373265006',
   'destinationLabel': 'Analgesic',
   'relationshipGroup': 0},
  {'typeId': '116680003',
   'typeLabel': 'Is a',
   'destinationId': '9680003',
   'destinationLabel': 'Cen

In [79]:
msgs = build_messages_from_iter(system, user, all_payloads[0])

KeyError: 'query_text'

In [43]:
####### POST COORD (PP)

### Example Concept Post Coordination -- Query: 'hypersensitivity to oxymorphone'

LLM Output: 

*   "ci_text":   "known hypersensitivity to oxymorphone",
*   "condition_text":    "hypersensitivity",
*   "substance_text":    "oxymorphone",
*   "severity_span":     null,
*   "course_span":   null,
*   "age_constraint":    null,
*   "population_text":   null,
*   "other_modifiers":   null

CI text matches --> No direct match

condition text matches: 

    252097006 - 'Hypersensitivity finding (finding)'
    473010000 - 'Hypersensitivity condition (finding)
    609433001 - Hypersensitivity disposition (finding)

Substance text matches: 

    24751001 - 'Oxymorphone (substance)'
    69899006 - 'Oxymorphone hydrochloride (substance)'
    725694003 - 'Oxabolone (substance)'

Proceeding with 609433001 - Hypersensitivity disposition (finding) for example purposes


In [63]:
# DOMAIN = pd.read_csv('snomed_int_source/der2_sssssssRefset_MRCMDomainSnapshot_INT_20260201.txt', sep='\t')
# DOMAIN = DOMAIN[DOMAIN['active']==1]
# ATTR_DOMAIN = pd.read_csv('snomed_int_source/der2_cissccRefset_MRCMAttributeDomainSnapshot_INT_20260201.txt', sep='\t')
# ATTR_DOMAIN = ATTR_DOMAIN[ATTR_DOMAIN['active']==1]
# ATTR_RANGE = pd.read_csv('snomed_int_source/der2_ssccRefset_MRCMAttributeRangeSnapshot_INT_20260201.txt', sep='\t')
# ATTR_RANGE = ATTR_RANGE[ATTR_RANGE['active']==1]
## Redoing this to keep it consistent with the US edition

In [100]:
ATTRIBUTE_TABLE = {
  "causative_agent": 246075003,   # Causative agent (attribute)
  "severity": 246112005,          # Severity (attribute)
  "clinical_course": 263502005,   # Clinical course (attribute)
}

In [12]:
# iMac ip - 139.52.39.136
#Snostorm port 8080
IMAC_IP = "139.52.39.136"
SNOW_PORT = 8080
TEST_URL = f"http://{IMAC_IP}:{SNOW_PORT}"
base = f"http://{IMAC_IP}:{SNOW_PORT}/MAIN/SNOMEDCT-US/concepts"

import requests
try:
    response = requests.get(TEST_URL, timeout=5)
    print("iMac is reachable:", response.status_code)
except requests.RequestException as e:
    print("Error reaching iMac:", e)

iMac is reachable: 200


In [140]:
base = f"http://{IMAC_IP}:{SNOW_PORT}/MAIN/SNOMEDCT-US/concepts"

def expand_ecl_to_concepts(ecl: str, base: str = base, limit: int = 10000) -> set[str]:
    """
    Use Snowstorm's ECL endpoint to expand the ECL to conceptIds.
    Much more robust than custom parsing.
    """
    if not isinstance(ecl, str) or not ecl.strip():
        return set()
    
    try:
        resp = requests.get(base,params={"ecl": ecl,"limit": limit},timeout=10)
        resp.raise_for_status()
        data = resp.json()
        return {item["conceptId"] for item in data["items"]}
    except Exception as e:
        print(f"ECL expansion failed for '{ecl}': {e}")
        return set()

In [18]:
import pandas as pd
from typing import Dict, Any, List, Tuple, Set

DOMAIN = pd.read_csv('snomed_us_source/der2_sssssssRefset_MRCMDomainSnapshot_US1000124_20250901.txt', sep='\t')
DOMAIN = DOMAIN[DOMAIN['active']==1]
ATTR_DOMAIN = pd.read_csv('snomed_us_source/der2_cissccRefset_MRCMAttributeDomainSnapshot_US1000124_20250901.txt', sep='\t')
ATTR_DOMAIN = ATTR_DOMAIN[ATTR_DOMAIN['active']==1]
ATTR_RANGE = pd.read_csv('snomed_us_source/der2_ssccRefset_MRCMAttributeRangeSnapshot_US1000124_20250901.txt', sep='\t')
ATTR_RANGE = ATTR_RANGE[ATTR_RANGE['active']==1]


In [19]:
def get_ancestors(concept_id: int, snomed_rel_df: pd.DataFrame) -> Set[int]:
    """
    Get all ancestor conceptIds for a given conceptId using the snomed_rel_df.
    Only follows active "Is a" relationships.
    """
    ancestors = set()
    to_visit = [concept_id]

    while to_visit:
        current = to_visit.pop()
        parents = snomed_rel_df[
            (snomed_rel_df['typeId'] == 116680003) &
            (snomed_rel_df['active'] == 1) &
            (snomed_rel_df.index == current) # index is sourceId
        ]['destinationId'].tolist()

        for p in parents:
            if p not in ancestors:
                ancestors.add(p)
                to_visit.append(p)

    return ancestors

In [20]:
def concept_matches_ecl(concept_id: int, ecl: str, base: str) -> bool:
    """
    Check if a concept satisfies an ECL without expanding the full set.
    """
    test_ecl = f"({ecl}) AND {concept_id}"
    
    try:
        resp = requests.get(
            base,
            params={"ecl": test_ecl, "limit": 1},
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json()
        return data.get("total", 0) > 0
    except Exception as e:
        print(f"ECL membership check failed: {e}")
        return False

In [ ]:
def get_domains_for_concept_snowstorm(concept_id: int, mrcm_domain_df: pd.DataFrame) -> list[int]:
    domains = []
    for _, row in mrcm_domain_df.iterrows():
        ec = row["domainConstraint"]
        tokens = ec.strip().split()
        if len(tokens) >= 2 and tokens[0] in ("<", "<<"):
            target = int(tokens[1]) 
        if concept_matches_ecl(concept_id, ec, base):
            domains.append(target)
    return domains

In [4]:
!nvidia-smi

Tue Mar 17 10:43:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.51.03              Driver Version: 575.51.03      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:27:00.0 Off |                    0 |
| N/A   26C    P0             73W /  400W |    1749MiB /  81920MiB |      4%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
!ps -u -p 2104266

USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
srv-xha+ 2104266  100  0.1 84306744 2173056 ?    Rl   07:35 170:22 python fine_t


In [ ]:
# def get_domains_for_concept(concept_id: int, snomed_rel_df: pd.DataFrame, mrcm_domain_df: pd.DataFrame) -> dict[dict]:
#     ancestors = get_ancestors(concept_id, snomed_rel_df)
#     domains = {}
#     for _, row in mrcm_domain_df.iterrows():
#         ec = row["domainConstraint"]  # e.g. "<< 404684003 |Clinical finding|"
#         # naive: handle patterns like "<< <SCTID>" or "< <SCTID>"
#         tokens = ec.strip().split()
#         if len(tokens) >= 2 and tokens[0] in ("<", "<<"):
#             target = int(tokens[1])  # SCTID
#             if target in ancestors:
#                 domains["target"] = target
#                 domains["data"] = (row.to_dict())
#     return domains

In [22]:
## attribute is referencedComponentId
def get_allowed_attributes_for_domain(domain_id: int, attr_domain_df: pd.DataFrame) -> list[int]:
    rows = attr_domain_df[attr_domain_df["domainId"] == domain_id]
    return [r["referencedComponentId"] for _, r in rows.iterrows()]

In [23]:
def get_range_constraints_for_attribute(attribute_id: int, attr_range_df: pd.DataFrame) -> list[str]:
    rows = attr_range_df[attr_range_df["referencedComponentId"] == attribute_id]
    return [r["rangeConstraint"] for _, r in rows.iterrows()]

In [ ]:
def get_allowed_value_concepts_for_attribute(attribute_id: int, attr_range_df: pd.DataFrame) -> set[int]:
    """
    Use MRCM attribute range to get the set of allowed conceptIds
    for a given attribute, using your minimal ECL expansion.
    """
    allowed = set()
    constraints = get_range_constraints_for_attribute(attribute_id, attr_range_df)  
    for ec in constraints:
        allowed |= expand_ecl_to_concepts(ec)  # ToDo: replace this with a direct ECL membership check if possible for efficiency
    return allowed

#### MRCM Attribute Range Reference Set


* referencedComponentId: 

        A reference to the SNOMED CT component to be included in the reference set. A reference to the SNOMED CT **attribute** concept to which the range defined by this member applies.

In [77]:
ATTR_RANGE[ATTR_RANGE['referencedComponentId'].isin(y)]

,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,rangeConstraint,attributeRule,ruleStrengthId,contentTypeId
7,9883dd72-0fdc-47bf-89d0-4f206ca7800c,20210731,1,900000000000012004,723562003,363713009,<< 260245000 |Finding value (qualifier value)|...,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
8,98984674-6f17-40fe-8859-5ad2d0f5c09d,20200731,1,900000000000012004,723562003,419066007,<< 419358007 |Subject of record or other provi...,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
12,a079f482-e7de-4fd8-b4f9-5c8b56798676,20220731,1,900000000000012004,723562003,363698007,<< 442083009 |Anatomical or acquired body stru...,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
14,a107baf4-2724-4ef2-acd1-8f153ed31c21,20200731,1,900000000000012004,723562003,363714003,<< 108252007 |Laboratory procedure (procedure)...,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
33,c7599c64-e54d-447e-a8ea-f051e19fe312,20170731,1,900000000000012004,723562003,246112005,<< 272141005 |Severities (qualifier value)|,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
36,cc2148e5-f5cd-4ee5-aca1-94dd30c2a0a5,20200731,1,900000000000012004,723562003,246075003,<< 105590001 |Substance (substance)| OR 138875...,(<< 272379006 |Event (event)| OR << 404684003 ...,723597001,723595009
46,e25ae83a-4da4-4c87-b254-f5399dd9b088,20230228,1,900000000000012004,723562003,719722006,<< 272379006 |Event (event)| OR << 404684003 |...,(<< 363787002 |Observable entity (observable e...,723597001,723596005
53,e8ec5f5c-a9e0-4241-912d-8ea9331cce91,20220930,1,900000000000012004,723562003,370135005,<< 1495041000004108 |Proliferation of neoplasm...,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
59,f481c372-2d43-4db9-becb-4f5cb530792a,20200731,1,900000000000012004,723562003,42752001,<< 272379006 |Event (event)| OR << 404684003 |...,(<< 272379006 |Event (event)| OR << 404684003 ...,723597001,723596005
63,f5f6382f-b32c-4792-bfdd-bc2f739dc57d,20200731,1,900000000000012004,723562003,47429007,<< 105590001 |Substance (substance)| OR 138875...,(<< 272379006 |Event (event)| OR << 404684003 ...,723597001,723595009


In [30]:
z = get_allowed_value_concepts_for_attribute(246075003, ATTR_RANGE)

#### For 609433001 - Hypersensitivity disposition (finding) the allowed attributes are: 

In [97]:
#     252097006 - 'Hypersensitivity finding (finding)'
#     473010000 - 'Hypersensitivity condition (finding)
#     609433001 - Hypersensitivity disposition (finding)

# Substance text matches: 

#     24751001 - 'Oxymorphone (substance)'
#     69899006 - 'Oxymorphone hydrochloride (substance)'
#     725694003 - 'Oxabolone (substance)'
# focus = 609433001
# focus = 41291007 # Angioedema (disorder)
focus = 716186003 # no known allergy (situation)

In [99]:
dom = get_domains_for_concept_snowstorm(focus, DOMAIN)
attr = get_allowed_attributes_for_domain(dom[0], ATTR_DOMAIN)
# print(f"Domains for concept {focus}: {dom}")
# print(f"Allowed attributes for domain {dom[0]}: {attr}")

In [92]:
dom

[]

In [96]:
ATTR_RANGE[ATTR_RANGE['referencedComponentId']==246454002]

,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,rangeConstraint,attributeRule,ruleStrengthId,contentTypeId
111,47e35159-6db8-41f2-a4cd-4fa79ea0e03b,20231001,1,900000000000012004,723562003,246454002,<< 282032007 |Periods of life (qualifier value)|,(<< 272379006 |Event (event)| OR << 404684003 ...,723597001,723596005


In [85]:
# allowed_codes = 
# ATTR_RANGE_filtered = ATTR_RANGE[(ATTR_RANGE['referencedComponentId'].isin(attr)) & (ATTR_RANGE['contentTypeId']==723595009)] 
ATTR_RANGE_filtered = ATTR_RANGE[(ATTR_RANGE['referencedComponentId'].isin(attr)) ] 
#723594008: precoordinated, 723595009: postcoordinated
ATTR_RANGE_filtered

,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,rangeConstraint,attributeRule,ruleStrengthId,contentTypeId
7,9883dd72-0fdc-47bf-89d0-4f206ca7800c,20210731,1,900000000000012004,723562003,363713009,<< 260245000 |Finding value (qualifier value)|...,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
8,98984674-6f17-40fe-8859-5ad2d0f5c09d,20200731,1,900000000000012004,723562003,419066007,<< 419358007 |Subject of record or other provi...,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
12,a079f482-e7de-4fd8-b4f9-5c8b56798676,20220731,1,900000000000012004,723562003,363698007,<< 442083009 |Anatomical or acquired body stru...,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
14,a107baf4-2724-4ef2-acd1-8f153ed31c21,20200731,1,900000000000012004,723562003,363714003,<< 108252007 |Laboratory procedure (procedure)...,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
33,c7599c64-e54d-447e-a8ea-f051e19fe312,20170731,1,900000000000012004,723562003,246112005,<< 272141005 |Severities (qualifier value)|,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
36,cc2148e5-f5cd-4ee5-aca1-94dd30c2a0a5,20200731,1,900000000000012004,723562003,246075003,<< 105590001 |Substance (substance)| OR 138875...,(<< 272379006 |Event (event)| OR << 404684003 ...,723597001,723595009
46,e25ae83a-4da4-4c87-b254-f5399dd9b088,20230228,1,900000000000012004,723562003,719722006,<< 272379006 |Event (event)| OR << 404684003 |...,(<< 363787002 |Observable entity (observable e...,723597001,723596005
53,e8ec5f5c-a9e0-4241-912d-8ea9331cce91,20220930,1,900000000000012004,723562003,370135005,<< 1495041000004108 |Proliferation of neoplasm...,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
59,f481c372-2d43-4db9-becb-4f5cb530792a,20200731,1,900000000000012004,723562003,42752001,<< 272379006 |Event (event)| OR << 404684003 |...,(<< 272379006 |Event (event)| OR << 404684003 ...,723597001,723596005
63,f5f6382f-b32c-4792-bfdd-bc2f739dc57d,20200731,1,900000000000012004,723562003,47429007,<< 105590001 |Substance (substance)| OR 138875...,(<< 272379006 |Event (event)| OR << 404684003 ...,723597001,723595009


In [81]:
attr_list = [a for a in attr]
ATTR_RANGE_filtered = ATTR_RANGE[(ATTR_RANGE['referencedComponentId'].isin(attr)) & (ATTR_RANGE['contentTypeId']==723594008)] 
# cross ref with concept_df to get labels
attr_concepts = concept_df[concept_df['conceptId'].isin(attr_list)][['conceptId', 'term']]
attr_concepts['ranges'] = attr_concepts['conceptId'].apply(lambda cid: get_range_constraints_for_attribute(cid, ATTR_RANGE_filtered))
attr_concepts['rule'] = attr_concepts['conceptId'].map(ATTR_RANGE_filtered.set_index('referencedComponentId')['attributeRule'])
attr_concepts

,conceptId,term,ranges,rule
618776,246075003,Causative agent (attribute),[<< 105590001 |Substance (substance)| OR << 26...,(<< 272379006 |Event (event)| OR << 404684003 ...
618818,246112005,Severity (attribute),[],NaN
619197,246454002,Occurrence (attribute),[],NaN
619200,246456000,Episodicity (attribute),[],NaN
628939,255234002,After (attribute),[],NaN
638326,263502005,Clinical course (attribute),[],NaN
663593,116676008,Associated morphology (attribute),[],NaN
665597,288556008,Before (attribute),[],NaN
743475,363698007,Finding site (attribute),[],NaN
743491,363713009,Has interpretation (attribute),[],NaN


In [ ]:
408729009 : {246090004= 51875005  , 408729009 = 415684004 }

In [181]:
from collections import defaultdict

IS_A = 116680003

def build_descendants_index(rels_df):
    children_by_parent = defaultdict(set)
    for index, row in rels_df[rels_df["active"] == 1].iterrows():
        if row["typeId"] != IS_A:
            continue
        parent = row["destinationId"]
        child = index
        children_by_parent[parent].add(child)
    return children_by_parent

CHILDREN_BY_PARENT = build_descendants_index(snomed_rel_df)

In [ ]:
def get_descendants_or_self(concept_id: int) -> set[str]:
    """Return concept_id and all active descendants via IS_A."""
    result = set()
    stack = [concept_id]
    while stack:
        cid = stack.pop()
        if cid in result:
            continue
        result.add(cid)
        for ch in CHILDREN_BY_PARENT.get(cid, []):
            if ch not in result:
                stack.append(ch)
    return result

In [ ]:
SYSTEM_PROMPT = """
You are a SNOMED CT mapping assistant.

Task:
1) Choose the single best focus concept from the provided FOCUS_CANDIDATES for the meaning of the QUERY text.
2) For each attribute listed in ATTRIBUTE_TABLE, choose the single best VALUE candidate from the provided candidate list for that attribute.
- If a candidate list is empty, output "N/A" for that attribute.
- You MUST ONLY select conceptIds that appear in the provided candidate lists.
- Do NOT invent any conceptIds.

Output ONLY minified JSON followed by <<END_JSON>>.
No markdown fences.

Schema:
{{
  "selected_focus_id": "<conceptId from FOCUS_CANDIDATES>",
  "fills": {{
    "<attribute_key>": "<conceptId from that attribute's candidates or N/A>",
    ...
  }}
}}
<<END_JSON>>
"""

USER_PROMPT = """
QUERY:
{span_text}

ATTRIBUTE_TABLE (attribute_key -> SNOMED attributeId):
{attribute_table_json}

FOCUS_CANDIDATES (choose 1):
{focus_candidates_block}

CANDIDATES FOR causative_agent (choose 1 or N/A):
{causative_candidates_block}

CANDIDATES FOR severity (choose 1 or N/A):
{severity_candidates_block}

CANDIDATES FOR clinical_course (choose 1 or N/A):
{course_candidates_block}
"""

#### Implementation: 

In [2]:
from VaxMapper.src.utils.snomed_utils import load_snomed_dataframes

In [3]:
sn = load_snomed_dataframes(snomed_source_dir="snomed_us_source")

In [139]:
sn['concept_df']

,conceptId,term,semantic_tag
534,127362006,Previous pregnancies (finding),finding
181775,125587004,Superficial injury (disorder),disorder
181830,125643001,Open wound (disorder),disorder
181852,125665001,Crushing injury (disorder),disorder
181853,125666000,Burn (disorder),disorder
...,...,...,...
1696617,900000000000547002,Relationship inactivation indicator attribute ...,foundation metadata concept
1696618,900000000000548007,Preferred (foundation metadata concept),foundation metadata concept
1696620,900000000000549004,Acceptable (foundation metadata concept),foundation metadata concept
1696623,900000000000550004,Definition (core metadata concept),core metadata concept


In [4]:
sn.keys()

dict_keys(['concept_df', 'synonym_df', 'terms_df', 'snomed_complete_df', 'domain', 'attr_domain', 'attr_range'])

In [5]:
ATTR = sn['attr_range']

In [9]:
ATTR[ATTR['referencedComponentId']==246061005]

,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,rangeConstraint,attributeRule,ruleStrengthId,contentTypeId


In [ ]:
<< 243796009 |Situation with explicit context (situation)|: [0..*] { [0..1] 408731000 |Temporal context| = << 410510008 |Temporal context value (qualifier value)| }

#### prefilter.py troubleshooting: 

In [2]:
from VaxMapper.src.utils.snomed_utils import (
    ATTRIBUTE_TABLE,
    base as SNOMED_BASE_URL,
    check_snomed_connection,
    concept_matches_any_ecl,
    get_range_constraints_for_attribute,
    load_snomed_dataframes,
    get_domains_for_concept_snowstorm,
    get_allowed_attributes_for_domain
)

In [3]:
check_snomed_connection(base_url=SNOMED_BASE_URL)

In [4]:
from prefilter import read_jsonl, collect_unique_candidate_ids,build_membership_for_attribute
from typing import Any, Dict, Iterable, List, Optional, Set, Tuple

In [5]:
mapped_jsonl = "results/20260303/mapped_hits.jsonl"
snomed_source_dir = "snomed_us_source"
snomed_df_aff = load_snomed_dataframes(snomed_source_dir=snomed_source_dir)

In [6]:
res = list(ATTRIBUTE_TABLE.values())

In [7]:
ATTR_RANGE = snomed_df_aff['attr_range']
ATTR_RANGE[ATTR_RANGE['referencedComponentId'].isin(res)]

,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,rangeConstraint,attributeRule,ruleStrengthId,contentTypeId
33,c7599c64-e54d-447e-a8ea-f051e19fe312,20170731,1,900000000000012004,723562003,246112005,<< 272141005 |Severities (qualifier value)|,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
36,cc2148e5-f5cd-4ee5-aca1-94dd30c2a0a5,20200731,1,900000000000012004,723562003,246075003,<< 105590001 |Substance (substance)| OR 138875...,(<< 272379006 |Event (event)| OR << 404684003 ...,723597001,723595009
71,fe14346d-fd26-49ab-a9db-ba74f8eca9ee,20200731,1,900000000000012004,723562003,246075003,<< 105590001 |Substance (substance)| OR << 260...,(<< 272379006 |Event (event)| OR << 404684003 ...,723597001,723594008
132,61826c65-bbcb-4a47-b0ed-29bb6d608b32,20170731,1,900000000000012004,723562003,263502005,<< 288524001 |Courses (qualifier value)|,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005


In [8]:
ATTR_RANGE

,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,rangeConstraint,attributeRule,ruleStrengthId,contentTypeId
0,81288567-57a9-49b1-b7f0-bf5979a6d289,20170731,1,900000000000012004,723562003,405814001,<< 442083009 |Anatomical or acquired body stru...,<< 71388002 |Procedure (procedure)|: [0..*] { ...,723597001,723596005
1,8163e3c0-b0c4-4c92-82e7-e93a23770b17,20230630,1,900000000000012004,723562003,246090004,<< 272379006 |Event (event)| OR << 363787002 |...,<< 413350009 |Finding with explicit context (s...,723597001,723595009
2,81f6d23b-66c7-46ce-b003-08380b9b236a,20200731,1,900000000000012004,723562003,118170007,<< 125676002 |Person (person)| OR << 133928008...,<< 123038009 |Specimen (specimen)|: [0..*] { [...,723597001,723596005
3,8a4b2de3-41cf-49a4-995e-9f80c71e0684,20200731,1,900000000000012004,723562003,732943007,< 105590001 |Substance (substance)|,<< 373873005 |Pharmaceutical / biologic produc...,723597001,723596005
4,903d4612-9e2e-4434-94e8-eaff214f4467,20170731,1,900000000000012004,723562003,425391005,<< 49062001 |Device (physical object)|,<< 71388002 |Procedure (procedure)|: [0..*] { ...,723597001,723596005
...,...,...,...,...,...,...,...,...,...,...
145,773c7698-80ab-4059-9da4-a55d0c88d0fd,20200731,1,900000000000012004,723562003,733928003,<< 123037004 |Body structure (body structure)|,<< 123037004 |Body structure (body structure)|...,723597001,723594008
146,79cb1aae-7943-40e8-81ae-125379645f9f,20170731,1,900000000000012004,723562003,424361007,<< 105590001 |Substance (substance)|,<< 71388002 |Procedure (procedure)|: [0..*] { ...,723597001,723596005
147,7b87904a-0156-418f-82e5-cc63d9ee034a,20180131,1,900000000000012004,723562003,736473005,< 736477006 |Dose form transformation (transfo...,<< 736542009 |Pharmaceutical dose form (dose f...,723597001,723596005
148,7d23e837-4e2b-45cd-b5c8-8ea9b51daae0,20170731,1,900000000000012004,723562003,363702006,<< 404684003 |Clinical finding (finding)| OR <...,<< 71388002 |Procedure (procedure)|: [0..*] { ...,723597001,723596005


In [26]:
out = []
for domain in get_domains_for_concept_snowstorm(1290126002, snomed_df_aff['domain']):
    print(f"Domain: {domain}")
    allowed_attrs = get_allowed_attributes_for_domain(domain, snomed_df_aff['attr_domain'])
    print(f"Allowed attributes for domain {domain}: {allowed_attrs}")
    for attr in allowed_attrs:
        allowed_values = get_range_constraints_for_attribute(attr, snomed_df_aff['attr_range'])
        print(f"Allowed values for attribute {attr}: {allowed_values}")
        # output loc of allowed values in ATTR_RANGE for reference. only get index value and append to out
        locs = snomed_df_aff['attr_range'][snomed_df_aff['attr_range']['referencedComponentId'] == attr].index.tolist()
        out.extend(locs)

Domain: 129125009
Allowed attributes for domain 129125009: [408730004, 363589002]
Allowed values for attribute 408730004: ['<< 288532009 |Context values for actions (qualifier value)|']
Allowed values for attribute 363589002: ['<< 363787002 |Observable entity (observable entity)| OR << 71388002 |Procedure (procedure)|', '<< 71388002 |Procedure (procedure)|']
Domain: 243796009
Allowed attributes for domain 243796009: [408731000, 408732007]
Allowed values for attribute 408731000: ['<< 410510008 |Temporal context value (qualifier value)|']
Allowed values for attribute 408732007: ['<< 125676002 |Person (person)|']


In [ ]:
<< 129125009 |Procedure with explicit context (situation)|: [0..*] { [0..1] 363589002 |Associated procedure| = (<< 363787002 |Observable entity (observable entity)| OR << 71388002 |Procedure (procedure)|) }

In [ ]:
<< 129125009 |Procedure with explicit context (situation)|: [0..*] { [0..1] 408730004 |Procedure context| = << 288532009 |Context values for actions (qualifier value)| }

In [ ]:
<< 243796009 |Situation with explicit context (situation)|: [0..*] { [0..1] 408731000 |Temporal context| = << 410510008 |Temporal context value (qualifier value)| }

In [ ]:
<< 129125009 |Procedure with explicit context (situation)|: [0..*] { [0..1] 363589002 |Associated procedure| = << 71388002 |Procedure (procedure)| }

In [ ]:
<< 243796009 |Situation with explicit context (situation)|: [0..*] { [0..1] 408732007 |Subject relationship context| = << 125676002 |Person (person)| }

In [1]:
# use out to filter ATTR_RANGE for relevant attributes
ATTR_RANGE[ATTR_RANGE.index.isin(out)]

NameError: name 'ATTR_RANGE' is not defined

In [19]:
mapped_rows = list(read_jsonl(mapped_jsonl))
pools = collect_unique_candidate_ids(mapped_rows)
attr_range_df = load_snomed_dataframes(snomed_source_dir=snomed_source_dir)["attr_range"]

In [22]:
DEFAULT_CONTENT_TYPE: Dict[str, Optional[int]] = {
    "causative_agent": 723594008,  # precoordinated only
    "severity": None,
    "clinical_course": None,
}

In [24]:
constraints: Dict[str, List[str]] = {}
for key, attr_id in ATTRIBUTE_TABLE.items():
    ctype = DEFAULT_CONTENT_TYPE.get(key)
    constraints[key] = [
        str(e).strip()
        for e in get_range_constraints_for_attribute(
            attribute_id=int(attr_id),
            attr_range_df=attr_range_df,
            content_type_id=ctype,
        )
        if str(e).strip()
    ]

In [25]:
constraints

{'causative_agent': ['<< 105590001 |Substance (substance)| OR << 260787004 |Physical object (physical object)| OR << 373873005 |Pharmaceutical / biologic product (product)| OR << 410607006 |Organism (organism)| OR << 78621006 |Physical force (physical force)|'],
 'severity': ['<< 272141005 |Severities (qualifier value)|'],
 'clinical_course': ['<< 288524001 |Courses (qualifier value)|']}

In [26]:
memberships: Dict[str, Dict[int, bool]] = {}
stats_by_attr: Dict[str, Dict[str, int]] = {}

In [40]:
test_s = list(pools["causative_agent"])[:50]

In [41]:
test_s

[1269547009,
 425984002,
 69632003,
 768000003,
 782336005,
 385024007,
 423936008,
 780501001,
 311722000,
 326058001,
 1237205008,
 776618005,
 860799000,
 295551000,
 709247000,
 260735003,
 776831003,
 74367007,
 230441003,
 322601004,
 54526002,
 267518003,
 85246009,
 392659002,
 216531002,
 292307003,
 255443008,
 763560005,
 780157000,
 376701008,
 290898004,
 1359954007,
 241959003,
 766460000,
 319996000,
 772391008,
 772604000,
 377340006,
 76923000,
 293798009,
 703824001,
 398672006,
 77136007,
 775717000,
 423461001,
 292389002,
 1148453008,
 26362002,
 1179386005,
 298746005]

In [42]:
key= "causative_agent"
membership, stats = build_membership_for_attribute(
    attribute_key=key,
    concept_ids=set(test_s),
    ecls=constraints.get(key, []),
    max_workers=max(1, 4),
    timeout=10,
    retries=max(1, 2),
)
memberships[key] = membership
stats_by_attr[key] = stats

[causative_agent] concepts=50 allowed=30 http_errors=0


In [19]:
mapped_rows[0]['severity_span_terms']

[]

In [7]:
ATTRIBUTE_TABLE

{'causative_agent': 246075003,
 'severity': 246112005,
 'clinical_course': 263502005}

In [15]:
len(all_payloads)

4

In [16]:
all_payloads[0]

{'SPL_SET_ID': '285e60f8-2404-47d0-a171-a4f075d6f418',
 'item_index': 0,
 'query_text': 'known hypersensitivity to oxymorphone',
 'span_text': 'known hypersensitivity to oxymorphone',
 'attribute_table_json': '{\n  "causative_agent": 246075003,\n  "severity": 246112005,\n  "clinical_course": 263502005\n}',
 'focus_candidates': [{'id': '252097006',
   'label': 'Hypersensitivity finding (finding)'},
  {'id': '473010000', 'label': 'Hypersensitivity condition (finding)'},
  {'id': '609433001', 'label': 'Hypersensitivity disposition (finding)'},
  {'id': '472963003', 'label': 'Hypersensitivity process (qualifier value)'},
  {'id': '421961002', 'label': 'Hypersensitivity reaction (disorder)'},
  {'id': '272401006', 'label': 'Hypersensitivity types (qualifier value)'},
  {'id': '21626009', 'label': 'Cutaneous hypersensitivity (disorder)'},
  {'id': '609405001',
   'label': 'Non-allergic hypersensitivity condition (finding)'},
  {'id': '609396006',
   'label': 'Non-allergic hypersensitivity di

In [13]:
all_payloads[0]['causative_candidates_raw']

[{'id': '24751001', 'label': 'Oxymorphone (substance)'},
 {'id': '370266008',
  'label': 'Product containing oxymorphone (medicinal product)'},
 {'id': '69899006', 'label': 'Oxymorphone hydrochloride (substance)'},
 {'id': '777032000',
  'label': 'Product containing only oxymorphone (medicinal product)'},
 {'id': '121420005', 'label': 'Oxymorphone measurement (procedure)'},
 {'id': '780099002',
  'label': 'Product containing only oxymorphone in oral dose form (medicinal product form)'},
 {'id': '772660006',
  'label': 'Product containing oxymorphone in oral dose form (medicinal product form)'},
 {'id': '424178008',
  'label': 'Product containing precisely oxymorphone 10 milligram/1 each conventional release oral tablet (clinical drug)'},
 {'id': '423970009',
  'label': 'Product containing precisely oxymorphone 5 milligram/1 each conventional release oral tablet (clinical drug)'},
 {'id': '725694003', 'label': 'Oxabolone (substance)'},
 {'id': '126101002', 'label': 'Oxymetholone (substa

In [12]:
all_payloads[0]['causative_candidates_block']

'[\n  {\n    "id": 24751001,\n    "label": "Oxymorphone (substance)"\n  },\n  {\n    "id": 370266008,\n    "label": "Product containing oxymorphone (medicinal product)"\n  },\n  {\n    "id": 69899006,\n    "label": "Oxymorphone hydrochloride (substance)"\n  },\n  {\n    "id": 777032000,\n    "label": "Product containing only oxymorphone (medicinal product)"\n  },\n  {\n    "id": 780099002,\n    "label": "Product containing only oxymorphone in oral dose form (medicinal product form)"\n  },\n  {\n    "id": 772660006,\n    "label": "Product containing oxymorphone in oral dose form (medicinal product form)"\n  },\n  {\n    "id": 424178008,\n    "label": "Product containing precisely oxymorphone 10 milligram/1 each conventional release oral tablet (clinical drug)"\n  },\n  {\n    "id": 423970009,\n    "label": "Product containing precisely oxymorphone 5 milligram/1 each conventional release oral tablet (clinical drug)"\n  },\n  {\n    "id": 725694003,\n    "label": "Oxabolone (substance)"\n

In [18]:
chats[0][1]

{'role': 'user',
 'content': '\nQUERY:\nknown hypersensitivity to oxymorphone\n\nATTRIBUTE_TABLE (attribute_key -> SNOMED attributeId):\n{\n  "causative_agent": 246075003,\n  "severity": 246112005,\n  "clinical_course": 263502005\n}\n\nFOCUS_CANDIDATES (choose 1):\n[\n  {\n    "id": "252097006",\n    "label": "Hypersensitivity finding (finding)"\n  },\n  {\n    "id": "473010000",\n    "label": "Hypersensitivity condition (finding)"\n  },\n  {\n    "id": "609433001",\n    "label": "Hypersensitivity disposition (finding)"\n  },\n  {\n    "id": "472963003",\n    "label": "Hypersensitivity process (qualifier value)"\n  },\n  {\n    "id": "421961002",\n    "label": "Hypersensitivity reaction (disorder)"\n  },\n  {\n    "id": "272401006",\n    "label": "Hypersensitivity types (qualifier value)"\n  },\n  {\n    "id": "21626009",\n    "label": "Cutaneous hypersensitivity (disorder)"\n  },\n  {\n    "id": "609405001",\n    "label": "Non-allergic hypersensitivity condition (finding)"\n  },\n  {\

In [ ]:
## Why doesn't the concept 1 to 1 match hypertension ?? Let's find out !. It's the BM25 check why later. 

In [46]:
from VaxMapper.src.utils.elastisearch_utils import (
    bulk_index,
    create_index,
    get_es_client,
    run_elasticsearch,
    stop_elasticsearch,
)

In [47]:
from hyb_mapper import load_snomed_frames, build_es_index,build_or_load_faiss_index

In [48]:
run_elasticsearch()
es = get_es_client()

Starting Elasticsearch from /data/wmanuel3/VaxMapperRepo/esdata/elasticsearch-9.0.0/bin/elasticsearch: on port 9200...
Elasticsearch is running: 9.0.0


In [49]:
concept_df, terms_df, snomed_complete_df = load_snomed_frames("snomed_us_source/sct2_Concept_Snapshot_US1000124_20250901.txt",
"snomed_us_source/sct2_Description_Snapshot-en_US1000124_20250901.txt",)
build_es_index(es, "snomed_ct_es_index", snomed_complete_df)

st_model, faiss_index = build_or_load_faiss_index(
            terms_df=terms_df,
            model_name="tavakolih/all-MiniLM-L6-v2-pubmed-full",
            device="cuda",
            index_path="results/snomed_terms_dense_test.bin",
            rebuild_index=True,
        )

concept_meta_df = concept_df.set_index("conceptId")

Deleting existing index: snomed_ct_es_index
Creating index 'snomed_ct_es_index'...
Failed to index: {'index': {'_index': 'snomed_ct_es_index', '_id': '260413007', 'status': 400, 'error': {'type': 'document_parsing_exception', 'reason': '[1:77] failed to parse: [1:80] Non-standard token \'NaN\': enable `JsonReadFeature.ALLOW_NON_NUMERIC_NUMBERS` to allow\n at [Source: (byte[])"{"conceptId":260413007,"preferredTerm":"None (qualifier value)","synonyms":[NaN],"semantic_tag":"qualifier value"}"; line: 1, column: 80]', 'caused_by': {'type': 'x_content_parse_exception', 'reason': '[1:80] Non-standard token \'NaN\': enable `JsonReadFeature.ALLOW_NON_NUMERIC_NUMBERS` to allow\n at [Source: (byte[])"{"conceptId":260413007,"preferredTerm":"None (qualifier value)","synonyms":[NaN],"semantic_tag":"qualifier value"}"; line: 1, column: 80]', 'caused_by': {'type': 'json_parse_exception', 'reason': 'Non-standard token \'NaN\': enable `JsonReadFeature.ALLOW_NON_NUMERIC_NUMBERS` to allow\n at [Source: (b

Batches:   0%|          | 0/3973 [00:00<?, ?it/s]

Index built with 1017048 vectors mapped to 382170 unique concept IDs.
Saving CPU index to 'results/snomed_terms_dense_test.bin'...
Index saved successfully to 'results/snomed_terms_dense_test.bin'.
Moving FAISS index to GPU (device 0)...
Index moved to GPU successfully.
Moving FAISS index to GPU (device 0)...
Index moved to GPU successfully.


In [50]:
from VaxMapper.src.utils.search_utils import search_query, encode_query, dense_candidates, bm25_candidates

In [51]:
query = "hypertension"
hits = search_query(
                query_text=query,
                model=st_model,
                faiss_index=faiss_index,
                concept_meta_df=concept_meta_df,
                es=es,
                bm25_index="snomed_ct_es_index",
                label_column="term",
                meta_id_column=None,
                bm25_text_field="all_terms",
                bm25_id_field="conceptId",
                bm25_label_field="preferredTerm",
                semantic_tag_column="semantic_tag",
                bm25_semantic_tag_field="semantic_tag",
                k_dense=50,
                k_bm25=50,
                k_final=20,
                ## gating mechanism (optional)
                k_probe_vectors = 200,
                k_probe_concepts = 50,
                probe_max_tags = 2,
                probe_min_top1_ratio= 0.45,
                probe_min_top1_margin= 5,
                probe_confidence_mode= "either",
                final_tag_gating= True,
                final_escape_hatch= 3,
                enable_semantic_tag_gating=False,
                normalize_query= True,
            )

In [52]:
q_vec = encode_query(st_model, query=query)

In [53]:
res = dense_candidates(faiss_index, q_vec,concept_meta_df,label_column="term")

In [54]:
res

[{'id': 38341003,
  'label': 'Hypertensive disorder, systemic arterial (disorder)',
  'score': 1.0000001192092896,
  'semantic_tag': 'disorder'},
 {'id': 429198000,
  'label': 'Exertional hypertension (disorder)',
  'score': 0.904859185218811,
  'semantic_tag': 'disorder'},
 {'id': 417312002,
  'label': 'Hypertension suspected (situation)',
  'score': 0.9031019806861877,
  'semantic_tag': 'situation'},
 {'id': 59621000,
  'label': 'Essential hypertension (disorder)',
  'score': 0.8941054940223694,
  'semantic_tag': 'disorder'},
 {'id': 89620005,
  'label': 'Hyperextension (finding)',
  'score': 0.8832396268844604,
  'semantic_tag': 'finding'},
 {'id': 1356877007,
  'label': 'Stable hypertension (disorder)',
  'score': 0.8820763826370239,
  'semantic_tag': 'disorder'},
 {'id': 162659009,
  'label': 'Hypertension resolved (finding)',
  'score': 0.8802430033683777,
  'semantic_tag': 'finding'},
 {'id': 170578008,
  'label': 'Poor hypertension control (finding)',
  'score': 0.8722240924835

In [55]:
resb = bm25_candidates(
    es=es,
    index="snomed_ct_es_index",
    query_text=query,
    text_field="all_terms",
    id_field="conceptId",
    label_field="preferredTerm",
    k=20,
)

In [56]:
resb

[{'id': 59621000,
  'label': 'Essential hypertension (disorder)',
  'score': 12.097736,
  'semantic_tag': 'disorder'},
 {'id': 23130000,
  'label': 'Paroxysmal hypertension (disorder)',
  'score': 11.880215,
  'semantic_tag': 'disorder'},
 {'id': 4210003,
  'label': 'Ocular hypertension (disorder)',
  'score': 11.858436,
  'semantic_tag': 'disorder'},
 {'id': 48194001,
  'label': 'Pregnancy-induced hypertension (disorder)',
  'score': 11.734827,
  'semantic_tag': 'disorder'},
 {'id': 445237003,
  'label': 'Portopulmonary hypertension (disorder)',
  'score': 11.705672,
  'semantic_tag': 'disorder'},
 {'id': 34742003,
  'label': 'Portal hypertension (disorder)',
  'score': 11.676662,
  'semantic_tag': 'disorder'},
 {'id': 70995007,
  'label': 'Pulmonary hypertension (disorder)',
  'score': 11.676662,
  'semantic_tag': 'disorder'},
 {'id': 762463000,
  'label': 'Diastolic hypertension co-occurrent with systolic hypertension (disorder)',
  'score': 11.657097,
  'semantic_tag': 'disorder'},